# NCKH XAU/USD — Toàn bộ quy trình trong một notebook

```
00 Tải Dukascopy → 01 Làm sạch → dukascopy_XAUUSD_<khung>_sach.csv → NB1 ∥ NB2 → NB3
```

| Phần | Nội dung | Đầu ra (trong `MyDrive/Data_NghienCuu`) |
|---|---|---|
| 00 | Tải XAUUSD 2015–2025 từ Dukascopy (time + OHLCV, volume thật) | `dukascopy_XAUUSD_<khung>.csv` |
| 01 | Làm sạch: ngày giờ, trùng lặp, khuyết thiếu, sai cấu trúc OHLC, kiểm tra chéo | `dukascopy_XAUUSD_<khung>_sach.csv` + báo cáo |
| NB1 | Chỉ báo kỹ thuật → **dự báo xu hướng** uptrend / sideway / downtrend (1 / 0 / −1) | `gold_price_technical_signal.csv` |
| NB2 | **XGBoost walk-forward 2020 → 2025** (mỗi năm dự báo bằng mô hình chỉ học dữ liệu trước năm đó) → **dự báo xu hướng** | `gold_model_signals.csv`, `danh_gia_walk_forward.csv` |
| NB3 | **9 chiến lược** × 5 nguồn (MA, RSI, MACD, TH, XGB) → lệnh **BUY (1) / SELL (0)** với SL:TP theo R:R; backtest kiểu MT5 2020–2025, kết quả từng năm, đối chứng ngẫu nhiên, đối đầu XGB với từng luật kỹ thuật | `backtest_comparison_report.csv`, `ket_qua_theo_nam.csv`, `doi_dau_xgb_vs_ky_thuat.csv` |

**Cách chạy:** chỉnh ô *Cấu hình chung* bên dưới rồi bấm **Thời gian chạy → Chạy tất cả**
(Run all) và cấp quyền Google Drive. NB1 và NB2 vẫn **độc lập** (cùng đọc tệp sạch, không
dùng kết quả của nhau) — trong bản gộp chúng chỉ chạy lần lượt.

- Đã tải dữ liệu từ trước thì giữ `BO_QUA_TAI_NEU_DA_CO = True`: phần 00 tự bỏ qua.
- `KHUNG_CHAY` là khung dùng cho NB1, NB2, NB3 (phải nằm trong `KHUNG_TAI`).
- Mỗi phần bắt đầu bằng một ô "ranh giới" xóa biến của phần trước, nên có thể chạy lại
  riêng từ đầu một phần bất kỳ (sau khi đã chạy ô *Cấu hình chung*).

In [ ]:
#@title Cấu hình chung
KHUNG_TAI = 'M30 H1 H4 D1'       #@param {type:'string'}
KHUNG_CHAY = 'H1'                #@param ['M30', 'H1', 'H4', 'D1']
BO_QUA_TAI_NEU_DA_CO = True      #@param {type:'boolean'}

import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CAU_HINH_THU_MUC = '/content/drive/MyDrive/Data_NghienCuu'
except ImportError:                                  # chạy trên máy tính
    CAU_HINH_THU_MUC = os.environ.get('NCKH_THU_MUC', '.')
os.makedirs(CAU_HINH_THU_MUC, exist_ok=True)
assert KHUNG_CHAY in KHUNG_TAI.split(), 'KHUNG_CHAY phải nằm trong KHUNG_TAI'
print('Thư mục dữ liệu:', CAU_HINH_THU_MUC)
print('Tải/làm sạch:', KHUNG_TAI, '| NB1–NB3 chạy trên khung:', KHUNG_CHAY)

---
# PHẦN 00 — Tải dữ liệu Dukascopy

Bỏ qua tự động nếu đã có đủ tệp và `BO_QUA_TAI_NEU_DA_CO = True`.

In [ ]:
# ── Ranh giới phần: xóa biến của phần trước (như mở notebook mới), giữ cấu hình chung
import matplotlib.pyplot as _plt
_plt.close('all')
_GIU = {'KHUNG_TAI', 'KHUNG_CHAY', 'BO_QUA_TAI_NEU_DA_CO', 'CAU_HINH_THU_MUC',
        'In', 'Out', 'get_ipython', 'exit', 'quit', 'display'}
for _t in [t for t in list(globals()) if not t.startswith('_') and t not in _GIU]:
    del globals()[_t]
print('Bắt đầu %s' % 'PHẦN 00 — Tải dữ liệu Dukascopy')

# 00 — Tải dữ liệu XAUUSD 2015–2025 từ Dukascopy (volume thật)

Chạy notebook này **trước** NB1 và NB2. Bấm **Run all**, cấp quyền Google Drive khi được hỏi.

- Tải nến M1 của từng ngày 01/01/2015 → 31/12/2025 từ Dukascopy, gộp lên **M30, H1, H4, D1**.
- Ghi vào `MyDrive/Data_NghienCuu/dukascopy_XAUUSD_<khung>.csv` — 6 cột `time, open, high, low, close, volume`, time theo UTC, giá BID, **volume thật** của Dukascopy.
- Khoảng 3.400 tệp ngày. Cứ 20 giây script in một dòng tiến độ: số ngày xong, **tốc độ**, **thời gian còn lại**, thời gian chờ mạng trung bình, số lần thử lại.
- **Bị ngắt vẫn chạy tiếp được:** tệp ngày để trên đĩa của Colab (ghi nhanh); mỗi tháng tải xong được lưu thành một tệp trong `MyDrive/Data_NghienCuu/_luu_thang_dukascopy`. Mất phiên hay bấm dừng thì **Run all** lại — các tháng đã xong không tải lại.
- Script tự kiểm tra **giá hợp lý** (vàng 2015–2025 trong 1.000–5.000 USD) và **giờ UTC** (phiên đầu tuần mở tối Chủ nhật 21–23 giờ UTC); sai thì dừng và báo.

- Script tự giãn nhịp tối đa `TOC_DO = 10` yêu cầu/giây (gọi dồn ~20 yêu cầu/giây thì Dukascopy từ chối sau khoảng 3.200 tệp). Toàn bộ mất khoảng 6 phút.

**Nếu bị từ chối giữa chừng** (báo "Máy chủ Dukascopy từ chối liên tục"): các tháng đã xong vẫn còn trên Drive. Đợi 5–10 phút rồi chạy lại ô tải — chỉ tải phần còn thiếu. Vẫn bị từ chối thì chọn **Thời gian chạy → Ngắt kết nối và xoá thời gian chạy** (đổi máy Colab) rồi Run all, và hạ `TOC_DO` xuống 5.

Sau khi xong, ở NB1 và NB2 điền ví dụ
`DUONG_DAN_DU_LIEU = '/content/drive/MyDrive/Data_NghienCuu/dukascopy_XAUUSD_H1.csv'`.

In [ ]:
%%writefile cao_du_lieu_dukascopy.py
# -*- coding: utf-8 -*-
"""Tải XAUUSD (time + OHLCV) từ Dukascopy, giai đoạn 01/01/2015 → 31/12/2025.

Tải nến M1 của TỪNG NGÀY từ datafeed.dukascopy.com rồi tự gộp lên khung cần. Mọi
khung đều phủ trọn 2015–2025, đi ra từ cùng một nguồn, và VOLUME LÀ VOLUME THẬT của
Dukascopy. Khoảng 3.400 tệp ngày nhỏ; có bộ nhớ đệm nên bị ngắt thì chạy lại sẽ tải tiếp.
Tệp ngày để ở --bo-dem (đĩa cục bộ, ghi nhanh); mỗi tháng tải xong được gộp thành một tệp
trong --luu-thang (trên Colab để trong Google Drive: chỉ khoảng 130 tệp, mất phiên vẫn còn).

CHẠY
  python cao_du_lieu_dukascopy.py                               # M30 H1 H4 D1
  python cao_du_lieu_dukascopy.py --khung M15 H1 D1 --gia mid

  Trên Google Colab dùng notebook "00_Tai_du_lieu_Dukascopy.ipynb" (bấm Run all).

  Nếu mạng chặn Dukascopy (DNS trả 127.0.0.1 — gặp với một số nhà mạng ở Việt Nam),
  script dừng và báo; khi đó hãy chạy trên Google Colab.

ĐẦU RA
  <thu_muc>/dukascopy_XAUUSD_<khung>.csv — đúng 6 cột: time, open, high, low, close, volume
  time là giờ MỞ nến theo UTC. Giá là giá BID (giống biểu đồ MT5), đổi bằng --gia ask|mid.
  D1 gộp theo phiên ngoại hối 22:00 → 22:00 UTC (không sinh nến rác ngày Chủ nhật).

TỰ KIỂM TRA (dừng và báo nếu không đạt, để không lặng lẽ ghi dữ liệu sai)
  · Giá hợp lý: vàng 2015–2025 nằm trong khoảng 1.000–5.000 USD. Sai hệ số chia giá
    (ví dụ chia 100 thay vì 1.000) sẽ cho giá 10.000+ và bị chặn ngay.
  · Giờ UTC: thị trường vàng mở lại vào tối Chủ nhật lúc 21:00–23:00 UTC. Nếu nến đầu
    tuần rơi vào giờ khác thì thời gian đang bị lệch múi giờ.
"""
from __future__ import annotations

import argparse
import lzma
import os
import random
import socket
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

GOC = Path(__file__).resolve().parent.parent
TU, DEN = date(2015, 1, 1), date(2025, 12, 31)
HE_SO_GIA = 1000            # XAUUSD: 3 chữ số thập phân → giá = số nguyên / 1000
GIA_HOP_LY = (1000, 5000)   # khoảng giá vàng giai đoạn 2015–2025 (USD/oz)
UA = {"User-Agent": "Mozilla/5.0"}
COT = ["time", "open", "high", "low", "close", "volume"]
# Chờ kết nối 10 giây, chờ dữ liệu 20 giây: kết nối treo thì bỏ sớm và thử lại, thay vì
# mất trọn 60 giây mỗi lần như bản trước (nguyên nhân chạy rất chậm trên Colab).
THOI_GIAN_CHO = (10, 20)
SO_LAN_THU = 8
NGHI_CO_SO = 1.5            # giây nghỉ trước lần thử lại đầu, sau đó gấp đôi (tối đa 30)
# Tối đa 10 yêu cầu/giây cho mọi luồng cộng lại: chạy thử trên Colab ở ~20 yêu cầu/giây thì
# sau ~3.200 tệp máy chủ bắt đầu từ chối liên tục.
TOC_DO_TOI_DA = 10
KIEM_SOM = 40               # sau 40 tệp tải về mà toàn rỗng → dừng và báo
IN_TIEN_DO = 20             # giây giữa hai lần in tiến độ

# Khung → quy tắc gộp của pandas. D1 gộp theo phiên ngoại hối 22:00 → 22:00 UTC:
# gộp theo mốc 00:00 sẽ sinh một nến D1 giả chỉ dài 1–2 giờ vào tối Chủ nhật.
KHUNG = {"M1": ("1min", None), "M5": ("5min", None), "M15": ("15min", None),
         "M30": ("30min", None), "H1": ("1h", None), "H4": ("4h", None),
         "D1": ("24h", "22h")}

DTYPE = np.dtype([("t", ">i4"), ("o", ">i4"), ("c", ">i4"), ("l", ">i4"), ("h", ">i4"), ("v", ">f4")])
_luong = threading.local()


def _phien():
    if not hasattr(_luong, "s"):
        _luong.s = requests.Session()
        _luong.s.headers.update(UA)
    return _luong.s


# ══════════════════════════════════════════════════════════ tải và giải mã
def kiem_tra_mang():
    try:
        ip = socket.gethostbyname("datafeed.dukascopy.com")
    except OSError as e:
        raise SystemExit("Không phân giải được datafeed.dukascopy.com (%s). Hãy chạy trên Google Colab." % e)
    if ip.startswith("127.") or ip == "0.0.0.0":
        raise SystemExit("Mạng này đang CHẶN Dukascopy: tên miền phân giải về %s.\n"
                         "→ Chạy trên Google Colab bằng notebook 00_Tai_du_lieu_Dukascopy.ipynb." % ip)


def giai_ma_ngay(noi_dung: bytes, ngay: date) -> pd.DataFrame:
    """Tệp BID/ASK_candles_min_1.bi5: LZMA, mỗi nến 24 byte big-endian
    (giây kể từ 00:00 UTC, open, close, low, high, volume)."""
    if not noi_dung:
        return pd.DataFrame()
    raw = lzma.decompress(noi_dung)
    a = np.frombuffer(raw[:len(raw) - len(raw) % 24], dtype=DTYPE)
    t = pd.Timestamp(ngay) + pd.to_timedelta(a["t"].astype("int64"), unit="s")
    d = pd.DataFrame({"time": t.astype("datetime64[ns]"),
                      "open": a["o"] / HE_SO_GIA, "high": a["h"] / HE_SO_GIA,
                      "low": a["l"] / HE_SO_GIA, "close": a["c"] / HE_SO_GIA,
                      "volume": a["v"].astype(float)})
    # Phút không có tick được Dukascopy điền nến phẳng với volume 0 (cả cuối tuần) → bỏ
    return d[d["volume"] > 0]


def _doc_duoc(noi_dung: bytes) -> bool:
    try:
        if noi_dung:
            lzma.decompress(noi_dung)
        return True
    except (lzma.LZMAError, EOFError):
        return False


def _ghi_an_toan(p: Path, noi_dung: bytes) -> None:
    """Ghi ra tệp tạm rồi đổi tên: bị ngắt giữa lúc ghi cũng không để lại tệp hỏng."""
    p.parent.mkdir(parents=True, exist_ok=True)
    tam = p.with_name(p.name + ".tam")
    tam.write_bytes(noi_dung)
    os.replace(tam, p)


class GioiHan:
    """Giãn đều các yêu cầu của mọi luồng để không vượt quá moi_giay yêu cầu/giây."""

    def __init__(self, moi_giay: float):
        self.khoang = 1.0 / moi_giay if moi_giay and moi_giay > 0 else 0.0
        self._khoa, self._ke = threading.Lock(), 0.0

    def cho(self, dung: threading.Event):
        if not self.khoang:
            return
        with self._khoa:
            bay_gio = time.monotonic()
            luot = max(bay_gio, self._ke)
            self._ke = luot + self.khoang
        if luot > bay_gio:
            dung.wait(luot - bay_gio)


class ThongKe:
    """Đếm số yêu cầu, số lần thử lại và thời gian chờ mạng (dùng chung giữa các luồng)."""

    def __init__(self, toc_do: float = 0):
        self.gioi_han = GioiHan(toc_do)
        self._khoa = threading.Lock()
        self.yeu_cau = self.thu_lai = self.da_tai = self.rong = 0
        self.giay_mang = 0.0
        self.nghi_van = set()           # ngày máy chủ trả 200 nhưng tệp rỗng cả 3 lần
        # Bật khi có lỗi hoặc bị ngắt: mọi luồng của lần chạy này đang nghỉ/thử lại dừng ngay
        self.dung = threading.Event()

    def cong(self, **kw):
        with self._khoa:
            for k, v in kw.items():
                setattr(self, k, getattr(self, k) + v)

    def them_nghi_van(self, ngay: date):
        with self._khoa:
            self.nghi_van.add(ngay)


def tai_ngay(ky_hieu: str, loai: str, ngay: date, bo_dem: Path, tk: ThongKe | None = None) -> bytes:
    tk = tk or ThongKe()
    p = bo_dem / ky_hieu / loai / ("%s.bi5" % ngay.isoformat())
    if p.exists():
        b = p.read_bytes()
        if _doc_duoc(b):
            return b
        p.unlink()                      # tệp đệm hỏng (bản cũ bị ngắt giữa lúc ghi) → tải lại
    url = ("https://datafeed.dukascopy.com/datafeed/%s/%04d/%02d/%02d/%s_candles_min_1.bi5"
           % (ky_hieu, ngay.year, ngay.month - 1, ngay.day, loai))      # tháng đếm từ 0
    so_rong, loi = 0, ""
    for lan in range(SO_LAN_THU):
        tk.gioi_han.cho(tk.dung)
        if tk.dung.is_set():
            raise RuntimeError("Đã dừng: %s" % url)
        t0 = time.monotonic()
        try:
            r = _phien().get(url, timeout=THOI_GIAN_CHO)
            loi = "HTTP %d" % r.status_code
        except requests.RequestException as e:
            r, loi = None, type(e).__name__
        tk.cong(yeu_cau=1, giay_mang=time.monotonic() - t0)
        if r is not None and r.status_code == 404:
            b = b""                                                      # ngày không có dữ liệu
            break
        if r is not None and r.status_code == 200:
            b = r.content
            if b:
                break
            # Ngày giao dịch luôn có tệp khác rỗng (cả phút không tick cũng được điền);
            # tệp rỗng thường do máy chủ đang giới hạn. Rỗng 3 lần thì chấp nhận nhưng đánh
            # dấu nghi vấn và KHÔNG lưu đệm, để lần chạy sau tải lại ngày này.
            so_rong += 1
            if so_rong >= 3:
                tk.them_nghi_van(ngay)
                tk.cong(da_tai=1, rong=1)
                return b
        # Lỗi mạng, 429/503 hoặc tệp rỗng: bỏ kết nối cũ (có thể đã chết), nghỉ rồi thử lại
        tk.cong(thu_lai=1)
        _luong.__dict__.pop("s", None)
        tk.dung.wait(min(30.0, NGHI_CO_SO * 2 ** lan) * (0.5 + random.random()))
    else:
        tk.dung.set()                   # báo các luồng khác dừng ngay, không tải thêm vòng nữa
        raise RuntimeError("Tải thất bại sau %d lần (lỗi cuối: %s): %s" % (SO_LAN_THU, loi, url))
    tk.cong(da_tai=1, rong=int(not b))
    _ghi_an_toan(p, b)
    return b


def _tep_thang(luu_thang: Path | None, ky_hieu: str, loai: str, ym) -> Path | None:
    return luu_thang / ("%s_%s_%04d-%02d.csv.gz" % (ky_hieu, loai, *ym)) if luu_thang else None


def _phut(giay: float) -> str:
    return "%d phút %02d giây" % divmod(int(giay), 60)


def m1_dukascopy(ky_hieu: str, loai: str, bo_dem: Path, so_luong: int,
                 luu_thang: Path | None = None, toc_do: float | None = None) -> pd.DataFrame:
    """Nến M1 cả giai đoạn. Tệp ngày lưu ở bo_dem; mỗi tháng tải xong được gộp thành
    một tệp trong luu_thang (nên để trên Google Drive: ít tệp, bị ngắt vẫn chạy tiếp)."""
    # Lấy thừa một ngày ở đầu để nến D1 phiên 22:00 đầu tiên đủ dữ liệu
    ngay_ds = [TU - timedelta(days=1) + timedelta(days=i)
               for i in range((DEN - TU).days + 2)]
    ngay_ds = [d for d in ngay_ds if d.weekday() != 5]                    # thứ Bảy không giao dịch
    luu_thang = Path(luu_thang) if luu_thang else None
    thang = {}
    for d in ngay_ds:
        thang.setdefault((d.year, d.month), []).append(d)

    phan, can_tai = {}, []
    for ym, ds in thang.items():
        p = _tep_thang(luu_thang, ky_hieu, loai, ym)
        if p is not None and p.exists():
            g = pd.read_csv(p, parse_dates=["time"])
            g["time"] = g["time"].astype("datetime64[ns]")         # cùng đơn vị với nến vừa tải
            phan[ym] = g
        else:
            can_tai += ds
    print("Tải %s %s M1: %d tệp ngày | %d/%d tháng đã có sẵn | cần tải %d ngày, %d luồng"
          % (ky_hieu, loai, len(ngay_ds), len(phan), len(thang), len(can_tai), so_luong), flush=True)

    tk, cho_gop = ThongKe(TOC_DO_TOI_DA if toc_do is None else toc_do), {}
    t0 = lan_in = time.monotonic()
    xong = 0
    ex = ThreadPoolExecutor(max_workers=so_luong)
    try:
        viec = {ex.submit(tai_ngay, ky_hieu, loai, d, bo_dem, tk): d for d in can_tai}
        for f in as_completed(viec):
            d = viec[f]
            ym = (d.year, d.month)
            cho_gop.setdefault(ym, {})[d] = giai_ma_ngay(f.result(), d)
            xong += 1
            if len(cho_gop[ym]) == len(thang[ym]):                       # đủ ngày của tháng
                g = [x for _, x in sorted(cho_gop.pop(ym).items()) if len(x)]
                phan[ym] = pd.concat(g, ignore_index=True) if g else pd.DataFrame(columns=COT)
                p = _tep_thang(luu_thang, ky_hieu, loai, ym)
                if p is not None and not tk.nghi_van.intersection(thang[ym]):
                    p.parent.mkdir(parents=True, exist_ok=True)
                    tam = p.with_name(p.name + ".tam")               # ghi tạm rồi đổi tên
                    phan[ym].to_csv(tam, index=False, compression="gzip")
                    os.replace(tam, p)
            if tk.da_tai >= KIEM_SOM and tk.rong == tk.da_tai:
                raise SystemExit("Máy chủ Dukascopy trả về toàn tệp rỗng (%d tệp đầu) — có thể đang bị "
                                 "chặn hoặc giới hạn. Đợi vài phút rồi chạy lại với --luong nhỏ hơn."
                                 % tk.da_tai)
            bay_gio = time.monotonic()
            if bay_gio - lan_in >= IN_TIEN_DO or xong == len(can_tai):
                lan_in = bay_gio
                toc_do = xong / max(bay_gio - t0, 1e-9)
                print("  %4d / %d ngày | %.1f ngày/giây | còn khoảng %s | chờ mạng TB %.1f giây/yêu cầu"
                      " | thử lại %d | tệp rỗng %d"
                      % (xong, len(can_tai), toc_do, _phut((len(can_tai) - xong) / max(toc_do, 1e-9)),
                         tk.giay_mang / max(tk.yeu_cau, 1), tk.thu_lai, tk.rong), flush=True)
    except RuntimeError as e:
        tk.dung.set()                                          # dừng ngay, không chờ hàng đợi
        ex.shutdown(wait=False, cancel_futures=True)
        con = len(thang) - len(phan)
        noi_luu = ("đã lưu %d/%d tháng vào %s" % (len(phan), len(thang), luu_thang) if luu_thang
                   else "các ngày đã tải nằm trong %s" % bo_dem)
        raise SystemExit(
            "\n⚠ Máy chủ Dukascopy từ chối liên tục — %s\n  Tiến độ: %s, còn %d tháng chưa xong.\n"
            "  → Đợi 5–10 phút rồi chạy lại: chỉ tải phần còn thiếu. Nếu vẫn bị từ chối, chọn\n"
            "    Thời gian chạy → Ngắt kết nối và xoá thời gian chạy (đổi máy Colab) rồi Run all."
            % (e, noi_luu, con)) from e
    except BaseException:
        tk.dung.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown()
    if can_tai:
        print("  Xong phần tải sau %s." % _phut(time.monotonic() - t0))
    if tk.nghi_van:
        ds = sorted(tk.nghi_van)
        print("  ⚠ %d ngày máy chủ trả tệp rỗng cả 3 lần (có thể đang bị giới hạn): %s%s\n"
              "    Các tháng chứa những ngày này chưa được lưu — chạy lại để tải lại riêng các ngày đó."
              % (len(ds), ", ".join(map(str, ds[:10])), " …" if len(ds) > 10 else ""))

    co = [phan[ym] for ym in sorted(phan) if len(phan[ym])]
    if not co:
        raise SystemExit("Không tải được nến nào — kiểm tra lại kết nối tới Dukascopy.")
    m = pd.concat(co, ignore_index=True)
    return m.drop_duplicates("time").sort_values("time").reset_index(drop=True)


# ══════════════════════════════════════════════════════════ tự kiểm tra
def tu_kiem_tra(m1: pd.DataFrame) -> None:
    """Chặn hai lỗi âm thầm: sai hệ số chia giá và lệch múi giờ."""
    trung_vi = float(m1["close"].median())
    thap, cao = float(m1["low"].min()), float(m1["high"].max())
    print("  Giá: trung vị %.2f | thấp nhất %.2f | cao nhất %.2f USD" % (trung_vi, thap, cao))
    if not (GIA_HOP_LY[0] * 0.8 <= thap and cao <= GIA_HOP_LY[1] * 1.2):
        raise SystemExit("⚠ Giá nằm ngoài khoảng hợp lý %s USD — hệ số chia giá (%d) có thể sai."
                         % (GIA_HOP_LY, HE_SO_GIA))

    # Nến đầu tiên của mỗi tuần (sau khoảng nghỉ cuối tuần > 24 giờ) phải mở tối Chủ nhật UTC
    nghi = m1["time"].diff() > pd.Timedelta(hours=24)
    dau_tuan = m1.loc[nghi, "time"]
    gio = dau_tuan.dt.hour.value_counts(normalize=True)
    dung = float(gio.reindex([21, 22, 23]).fillna(0).sum())
    chu_nhat = float((dau_tuan.dt.dayofweek == 6).mean())
    print("  Giờ mở cửa đầu tuần (UTC): %s | %.1f%% vào Chủ nhật 21–23 giờ"
          % (", ".join("%dh:%.0f%%" % (h, 100 * v) for h, v in gio.head(4).items()), 100 * dung))
    if dung < 0.8 or chu_nhat < 0.8:
        raise SystemExit("⚠ Phiên đầu tuần không mở vào tối Chủ nhật 21–23 giờ UTC — thời gian "
                         "có thể đang lệch múi giờ.")
    print("  ✓ Giá hợp lý và thời gian đúng UTC.")


# ══════════════════════════════════════════════════════════ gộp và ghi
def gop(m: pd.DataFrame, khung: str) -> pd.DataFrame:
    quy_tac, lech = KHUNG[khung]
    if khung == "M1":
        return m.copy()
    g = m.set_index("time").resample(quy_tac, offset=lech) if lech else \
        m.set_index("time").resample(quy_tac)
    d = g.agg({"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"})
    return d.dropna(subset=["close"]).reset_index()


def cat_giai_doan(d: pd.DataFrame) -> pd.DataFrame:
    d = d[(d["time"] >= pd.Timestamp(TU)) & (d["time"] < pd.Timestamp(DEN) + pd.Timedelta(days=1))]
    return d.reset_index(drop=True)


def kiem_tra(d: pd.DataFrame, khung: str) -> None:
    sai = ((d["high"] < d[["open", "close"]].max(axis=1)) |
           (d["low"] > d[["open", "close"]].min(axis=1))).sum()
    thieu = d[["open", "high", "low", "close", "volume"]].isna().any(axis=1).sum()
    print("  %-3s %9d nến | %s → %s | sai hình học %d | thiếu giá trị %d | volume trung vị %.2f"
          % (khung, len(d), d["time"].min(), d["time"].max(), sai, thieu, d["volume"].median()))
    so_nam = d.groupby(d["time"].dt.year).size()
    print("      Số nến mỗi năm: " + ", ".join("%d:%d" % kv for kv in so_nam.items()))


def chay(khung_ds, gia="bid", thu_muc=None, bo_dem=None, so_luong=8, luu_thang=None, toc_do=None):
    thu_muc = Path(thu_muc) if thu_muc else GOC / "data" / "raw"
    thu_muc.mkdir(parents=True, exist_ok=True)
    bo_dem = Path(bo_dem) if bo_dem else thu_muc / "_bo_dem_dukascopy"
    kiem_tra_mang()

    if gia == "mid":
        b = m1_dukascopy("XAUUSD", "BID", bo_dem, so_luong, luu_thang, toc_do)
        a = m1_dukascopy("XAUUSD", "ASK", bo_dem, so_luong, luu_thang, toc_do)
        m1 = b.merge(a, on="time", suffixes=("_b", "_a"))
        m1 = pd.DataFrame({"time": m1["time"],
                           **{k: (m1[k + "_b"] + m1[k + "_a"]) / 2
                              for k in ("open", "high", "low", "close")},
                           "volume": m1["volume_b"]})
    else:
        m1 = m1_dukascopy("XAUUSD", gia.upper(), bo_dem, so_luong, luu_thang, toc_do)
    print("Đã có %d nến M1, %s → %s" % (len(m1), m1["time"].min(), m1["time"].max()))
    tu_kiem_tra(m1)

    print("\nGộp khung và ghi tệp (%s → %s):" % (TU, DEN))
    ra = []
    for k in khung_ds:
        d = cat_giai_doan(gop(m1, k))
        d["volume"] = d["volume"].round(4)
        kiem_tra(d, k)
        p = thu_muc / ("dukascopy_XAUUSD_%s.csv" % k)
        d[["time", "open", "high", "low", "close", "volume"]].to_csv(p, index=False)
        print("      → %s" % p)
        ra.append(p)
    return ra


def main(argv=None):
    ap = argparse.ArgumentParser(description="Tải XAUUSD 2015–2025 (time + OHLCV) từ Dukascopy")
    ap.add_argument("--khung", nargs="+", default=["M30", "H1", "H4", "D1"], choices=list(KHUNG))
    ap.add_argument("--gia", choices=["bid", "ask", "mid"], default="bid")
    ap.add_argument("--thu-muc", default=None, help="thư mục ghi tệp, mặc định data/raw")
    ap.add_argument("--bo-dem", default=None, help="thư mục lưu tệp ngày đã tải (nên để ở đĩa cục bộ)")
    ap.add_argument("--luu-thang", default=None,
                    help="thư mục lưu mỗi tháng một tệp (trên Colab nên để trong Google Drive)")
    ap.add_argument("--luong", type=int, default=8, help="số luồng tải song song")
    ap.add_argument("--toc-do", type=float, default=TOC_DO_TOI_DA,
                    help="tối đa bao nhiêu yêu cầu/giây (cộng mọi luồng); 0 = không giới hạn")
    a = ap.parse_args(argv)
    chay(a.khung, a.gia, a.thu_muc, a.bo_dem, a.luong, a.luu_thang, a.toc_do)


if __name__ == "__main__":
    main()

In [ ]:
import os
KHUNG = KHUNG_TAI
THU_MUC = CAU_HINH_THU_MUC
tep_can = [os.path.join(THU_MUC, 'dukascopy_XAUUSD_%s.csv' % k) for k in KHUNG.split()]
if BO_QUA_TAI_NEU_DA_CO and all(os.path.exists(p) for p in tep_can):
    print('Đã có đủ tệp %s trong %s → bỏ qua bước tải.' % (KHUNG, THU_MUC))
else:
    BO_DEM = '/content/_bo_dem_dukascopy'
    get_ipython().system('mkdir -p "%s" && cp -rn "%s/_bo_dem_dukascopy/." "%s/" 2>/dev/null; true'
                         % (BO_DEM, THU_MUC, BO_DEM))
    get_ipython().system('python cao_du_lieu_dukascopy.py --khung %s --thu-muc "%s" --bo-dem "%s" '
                         '--luu-thang "%s/_luu_thang_dukascopy" --luong 8 --toc-do 10'
                         % (KHUNG, THU_MUC, BO_DEM, THU_MUC))
    thieu = [os.path.basename(p) for p in tep_can if not os.path.exists(p)]
    if thieu:
        raise RuntimeError('Tải chưa xong (%s). Xem thông báo phía trên, đợi vài phút rồi chạy lại '
                           'ô này — chỉ tải phần còn thiếu.' % thieu)

In [ ]:
import pandas as pd
for k in KHUNG.split():
    d = pd.read_csv(f'{THU_MUC}/dukascopy_XAUUSD_{k}.csv', parse_dates=['time'])
    print(f"{k:>3}: {len(d):>7} nến | {d.time.min()} → {d.time.max()} | volume trung vị {d.volume.median():.2f}")

---
# PHẦN 01 — Làm sạch dữ liệu

Làm sạch mọi khung trong `KHUNG_TAI`, ghi `dukascopy_XAUUSD_<khung>_sach.csv`.

In [ ]:
# ── Ranh giới phần: xóa biến của phần trước (như mở notebook mới), giữ cấu hình chung
import matplotlib.pyplot as _plt
_plt.close('all')
_GIU = {'KHUNG_TAI', 'KHUNG_CHAY', 'BO_QUA_TAI_NEU_DA_CO', 'CAU_HINH_THU_MUC',
        'In', 'Out', 'get_ipython', 'exit', 'quit', 'display'}
for _t in [t for t in list(globals()) if not t.startswith('_') and t not in _GIU]:
    del globals()[_t]
print('Bắt đầu %s' % 'PHẦN 01 — Làm sạch dữ liệu')

# 01 — Làm sạch dữ liệu XAUUSD (Dukascopy 2015–2025)

Chạy **sau** notebook 00 và **trước** NB1, NB2. Bấm **Run all**.

```
00 Tải Dukascopy → 01 Làm sạch → dukascopy_XAUUSD_<khung>_sach.csv → NB1 ∥ NB2 → NB3
```

NB1 và NB2 cùng đọc một tệp đã làm sạch, nên so sánh backtest ở NB3 là công bằng.

| Bước | Nội dung | Cách xử lý |
|---|---|---|
| 1 | Chuẩn hóa ngày giờ | Đưa về UTC không múi giờ, sắp xếp, căn nến về đúng mốc của khung, bỏ nến thứ Bảy, thêm cột `ngay_giao_dich` |
| 2 | Bản ghi trùng lặp | Trùng hoàn toàn → bỏ; cùng thời điểm khác giá → giữ bản đầu, ghi báo cáo |
| 3a | Ô trống / giá không hợp lệ | **Bỏ dòng** (không lấy Close thay O/H/L), chỗ đó thành khoảng trống được gắn cờ |
| 3b | Thị trường đóng cửa | Cuối tuần, nghỉ hằng ngày ~21–22 giờ UTC, ngày lễ → **không điền, chỉ đánh dấu** |
| 3c | Mất nến giữa phiên | **Không điền**, gắn cờ `khoang_trong = 1`, liệt kê trong báo cáo |
| 4 | Nến sai cấu trúc OHLC | Sửa `High = max(O,H,L,C)`, `Low = min(O,H,L,C)`, gắn cờ `da_sua = 1` |
| 6 | Kiểm tra chéo và đầu ra | Gộp khung nhỏ lên khung lớn phải khớp; ghi tệp sạch, báo cáo, biểu đồ |

Không có bước nào **tự tạo ra giá**: dữ liệu chỉ bị bỏ dòng hỏng, sửa râu nến sai
cấu trúc, và thêm các cột đánh dấu để minh bạch.

In [ ]:
import os
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil.easter import easter
from pandas.tseries.holiday import USFederalHolidayCalendar

try:                                   # Trên Colab: đọc/ghi Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    THU_MUC = '/content/drive/MyDrive/Data_NghienCuu'
except ImportError:                    # Chạy trên máy tính
    THU_MUC = CAU_HINH_THU_MUC

KHUNG_DS = ['M1', 'M5', 'M15', 'M30', 'H1', 'H4', 'D1']
BUOC = {'M1': pd.Timedelta('1min'), 'M5': pd.Timedelta('5min'), 'M15': pd.Timedelta('15min'),
        'M30': pd.Timedelta('30min'), 'H1': pd.Timedelta('1h'), 'H4': pd.Timedelta('4h'),
        'D1': pd.Timedelta('1D')}
COT_GIA = ['open', 'high', 'low', 'close']
COT = COT_GIA + ['volume']
print('Thư mục lưu kết quả:', THU_MUC)

## Chọn dữ liệu cần làm sạch

Chọn ở ô dưới (bảng bên phải ô trên Colab):

- **`NGUON = 'tai_len'`** — chạy ô sẽ hiện nút **Choose Files / Chọn tệp**: chọn một hoặc
  nhiều tệp `.csv` / `.xlsx` trên máy tính (giữ Ctrl để chọn nhiều tệp).
- **`NGUON = 'drive'`** — dùng tệp có sẵn trong `MyDrive/Data_NghienCuu`. Gõ tên tệp vào
  `TEN_TEP_DRIVE` (nhiều tệp cách nhau dấu phẩy); để trống thì lấy mọi tệp
  `dukascopy_XAUUSD_*.csv` do notebook 00 tải về. Ô sẽ in danh sách tệp đang có trên Drive.

Tên tệp đặt thế nào cũng được: **khung thời gian tự nhận ra từ dữ liệu**. Tên cột có thể là
`time/Date/Datetime/Gmt time`, `Open/High/Low/Close`, `Volume/Tick volume`… Kết quả lưu vào
Drive thành `<tên gốc>_sach.csv`; bật `TAI_VE_MAY` để tải luôn về máy.

In [ ]:
#@title Chọn dữ liệu cần làm sạch
NGUON = 'drive'          # bản gộp: dùng tệp phần 00 vừa tải
TEN_TEP_DRIVE = ', '.join('dukascopy_XAUUSD_%s.csv' % k for k in KHUNG_TAI.split())
TAI_VE_MAY = False       #@param {type:'boolean'}

import glob
DUOI_HOP_LE = ('.csv', '.txt', '.xlsx', '.xls')

def tep_tren_drive():
    return sorted(os.path.basename(p) for p in glob.glob(os.path.join(THU_MUC, '*'))
                  if p.lower().endswith(DUOI_HOP_LE) and '_sach' not in os.path.basename(p)
                  and not os.path.basename(p).startswith(('bao_cao', 'danh_sach', 'kiem_tra')))

try:
    from google.colab import files as _files
except ImportError:
    _files = None

if NGUON == 'tai_len' and _files is not None:
    thu_muc_tai = '/content/tai_len'
    os.makedirs(thu_muc_tai, exist_ok=True)
    print('Bấm "Choose Files" và chọn tệp dữ liệu vàng cần làm sạch:')
    da_tai = _files.upload()
    TEP_CHON = []
    for ten, noi_dung in da_tai.items():
        p = os.path.join(thu_muc_tai, ten)
        with open(p, 'wb') as f:
            f.write(noi_dung)
        TEP_CHON.append(p)
else:
    if _files is None and os.environ.get('NCKH_TEP'):        # chạy trên máy tính
        TEP_CHON = [p.strip() for p in os.environ['NCKH_TEP'].split(';') if p.strip()]
    else:
        co_san = tep_tren_drive()
        print('Tệp đang có trong %s:' % THU_MUC)
        for x in co_san:
            print('  ·', x)
        if TEN_TEP_DRIVE.strip():
            TEP_CHON = [os.path.join(THU_MUC, x.strip()) for x in TEN_TEP_DRIVE.split(',') if x.strip()]
        else:
            TEP_CHON = [os.path.join(THU_MUC, x) for x in co_san if x.startswith('dukascopy_XAUUSD_')]

thieu = [p for p in TEP_CHON if not os.path.exists(p)]
if thieu:
    raise FileNotFoundError('Không thấy tệp: %s' % thieu)
if not TEP_CHON:
    raise ValueError('Chưa chọn tệp nào để làm sạch.')
print('\nSẽ làm sạch %d tệp:' % len(TEP_CHON))
for p in TEP_CHON:
    print('  ·', os.path.basename(p))

## Bước 1 — Chuẩn hóa ngày giờ

- Đọc cột `time` ở mọi dạng (có hoặc không kèm múi giờ) và đưa về **UTC, không kèm múi giờ**.
  Dukascopy vốn đã là UTC nên không phải xử lý giờ mùa hè.
- **Căn mốc:** nến M30 phải rơi vào phút :00/:30, H1 vào :00, H4 vào 0/4/8/12/16/20 giờ,
  D1 vào giờ mở phiên (22:00 UTC với Dukascopy — tự nhận ra từ dữ liệu). Nến lệch mốc được
  đưa về mốc đầu nến và đếm vào báo cáo.
- **Bỏ nến thứ Bảy** — thị trường vàng đóng cửa.
- Thêm cột **`ngay_giao_dich`**: phiên vàng mở lúc 22:00–23:00 UTC tối hôm trước, nên nến
  22:00 Chủ nhật thuộc phiên thứ Hai.

In [ ]:
BI_DANH = {'time': ['time', 'datetime', 'date time', 'timestamp', 'date', 'gmt time', 'local time',
                    'time (utc)', 'open time', 'thoi gian'],
           'open': ['open', 'o'], 'high': ['high', 'h'], 'low': ['low', 'l'],
           'close': ['close', 'c', 'adj close', 'price', 'last'],
           'volume': ['volume', 'vol', 'tick volume', 'tickvol', 'real volume', 'volume ']}


def doc_tep(p):
    if p.lower().endswith(('.xlsx', '.xls')):
        d = pd.read_excel(p)
    else:
        d = pd.read_csv(p)
        if d.shape[1] == 1:                                  # tách bằng tab hoặc ';'
            d = pd.read_csv(p, sep=None, engine='python')
    chuan = {' '.join(str(c).strip().lower().replace('<', ' ').replace('>', ' ').replace('_', ' ').split()): c
             for c in d.columns}
    ra = pd.DataFrame(index=d.index)
    for cot, ds in BI_DANH.items():
        goc = next((chuan[x] for x in ds if x in chuan), None)
        if goc is None and cot == 'volume':
            print('  ⚠ Tệp không có cột volume → volume = 0 (giá không bị ảnh hưởng).')
            ra['volume'] = 0.0
        elif goc is None:
            raise ValueError('Tệp %s không có cột %s. Các cột đang có: %s' % (p, cot, list(d.columns)))
        else:
            ra[cot] = d[goc]
    return ra


def nhan_dien_khung(d):
    t = doc_thoi_gian(d['time']).dropna().sort_values()
    buoc = t.diff().dropna()
    buoc = buoc[buoc > pd.Timedelta(0)].mode().iloc[0]
    return min(BUOC, key=lambda k: abs(np.log(BUOC[k] / buoc)))


def doc_thoi_gian(s):
    """Mọi dạng thời gian → UTC. 02/01/2025 có thể là ngày-trước hoặc tháng-trước: thử cả
    hai, chọn cách đọc được nhiều dòng nhất và cho thời gian tăng dần nhiều nhất."""
    if pd.api.types.is_numeric_dtype(s):                        # số giây / mili-giây Unix
        don_vi = 'ms' if s.abs().median() > 1e11 else 's'
        return pd.to_datetime(s, unit=don_vi, errors='coerce', utc=True)
    s = s.astype(str).str.strip()
    ung_vien = [pd.to_datetime(s, errors='coerce', utc=True, format='mixed')]
    if s.str.contains('/').mean() > 0.5:
        ung_vien.append(pd.to_datetime(s, errors='coerce', utc=True, format='mixed', dayfirst=True))
    diem = lambda t: (t.notna().mean(), (t.diff().dt.total_seconds() > 0).mean())
    return max(ung_vien, key=diem)


def chuan_hoa_thoi_gian(d, khung, nk):
    t = doc_thoi_gian(d['time'])
    d['time'] = t.dt.tz_localize(None).astype('datetime64[ns]')
    nk['thoi_gian_loi'] = int(d['time'].isna().sum())
    d = d.dropna(subset=['time'])

    # Mốc của khung. D1: mốc mở phiên phổ biến nhất (Dukascopy: 22:00 UTC)
    lech = (d['time'] - d['time'].dt.floor('D')).mode().iloc[0] if khung == 'D1' else pd.Timedelta(0)
    moc = (d['time'] - lech).dt.floor(BUOC[khung]) + lech
    nk['lech_moc'] = int((moc != d['time']).sum())
    nk['moc_mo_nen'] = str(lech)[-8:] if khung == 'D1' else ''
    d['time'] = moc
    return d.sort_values('time', kind='stable').reset_index(drop=True), lech


def bo_thu_bay(d, nk):
    thu_bay = d['time'].dt.dayofweek == 5
    nk['thu_bay'] = int(thu_bay.sum())
    return d[~thu_bay].reset_index(drop=True)


def them_ngay_giao_dich(d):
    d['ngay_giao_dich'] = (d['time'] + pd.Timedelta(hours=2)).dt.strftime('%Y-%m-%d')
    return d

## Bước 2 — Bản ghi trùng lặp

- **Trùng hoàn toàn** (cùng thời điểm, cùng giá, cùng volume): bỏ bản sau.
- **Cùng thời điểm nhưng khác giá:** giữ bản đầu tiên, bỏ bản sau, liệt kê trong báo cáo.

Dòng hỏng được bỏ **trước** bước này (bước 3a), để một bản trùng hỏng không đẩy bản
đúng ra ngoài.

In [ ]:
def bo_trung(d, nk):
    hoan_toan = d.duplicated(subset=['time'] + COT, keep='first')
    nk['trung_hoan_toan'] = int(hoan_toan.sum())
    d = d[~hoan_toan]
    khac_gia = d.duplicated(subset=['time'], keep='first')
    nk['trung_thoi_gian_khac_gia'] = int(khac_gia.sum())
    ds = d[d['time'].isin(d.loc[khac_gia, 'time'])]
    if len(ds):
        print('  Cùng thời điểm khác giá (giữ dòng đầu):')
        print(ds.head(10).to_string(index=False))
    return d[~khac_gia].reset_index(drop=True)

## Bước 3 — Dữ liệu khuyết thiếu

**3a. Ô trống hoặc giá không hợp lệ** (thiếu một trong O/H/L/C/V, giá ≤ 0, volume âm):
**bỏ cả dòng**. Không lấy Close thay cho O/H/L vì như vậy là tự tạo giá. Chỗ bị bỏ trở
thành một khoảng trống và được gắn cờ ở 3c.

**3b, 3c. Khoảng trống giữa hai nến liền nhau.** Vàng **không giao dịch 24/7**, nên
khoảng trống được phân loại chứ không điền:

| Loại (`loai_khoang_trong`) | Nhận biết | Xử lý |
|---|---|---|
| `cuoi_tuan` | Khoảng thiếu có chứa ngày thứ Bảy | Chỉ đánh dấu |
| `ngay_le` | Khoảng thiếu chạm ngày lễ: 24–26/12, 31/12–2/1, Thứ Sáu Tuần Thánh, ngày lễ liên bang Mỹ, Thứ Sáu sau Lễ Tạ ơn | Chỉ đánh dấu |
| `nghi_hang_ngay` | Bắt đầu từ 20:00 UTC trở đi và dài không quá 3 giờ | Chỉ đánh dấu |
| `bat_thuong` | Mọi khoảng còn lại (mất nến giữa phiên) | Gắn cờ `khoang_trong = 1`, liệt kê trong báo cáo |

Nhãn được ghi vào **nến ngay sau khoảng trống**, kèm `so_nen_thieu`. **Không chèn nến
giả**: nến phẳng hay nội suy ở giờ đóng cửa sẽ tạo lợi suất bằng 0 không có thật, làm
lệch độ biến động, các chỉ báo và nhãn Triple Barrier.

In [ ]:
def bo_o_trong(d, nk):
    for c in COT:
        d[c] = pd.to_numeric(d[c], errors='coerce')
    hong = d[COT].isna().any(axis=1) | (d[COT_GIA] <= 0).any(axis=1) | (d['volume'] < 0)
    nk['o_trong_gia_khong_hop_le'] = int(hong.sum())
    return d[~hong].reset_index(drop=True)


def ngay_le(nam_tu, nam_den):
    le = set()
    for y in range(nam_tu, nam_den + 1):
        le |= {date(y, 12, 24), date(y, 12, 25), date(y, 12, 26), date(y, 12, 31),
               date(y, 1, 1), date(y, 1, 2)}
        le.add(easter(y) - timedelta(days=2))                   # Thứ Sáu Tuần Thánh
    lich = USFederalHolidayCalendar().holidays('%d-01-01' % nam_tu, '%d-12-31' % nam_den)
    for x in lich.date:
        le.add(x)
        if x.month == 11 and 22 <= x.day <= 28:                 # Lễ Tạ ơn → cả Thứ Sáu kế tiếp
            le.add(x + timedelta(days=1))
    return le


def phan_loai_khoang(d, khung, le):
    buoc = BUOC[khung]
    t = d['time']
    kc = t.diff()
    loai_cot = np.full(len(d), '', dtype=object)
    so_thieu_cot = np.zeros(len(d), dtype=int)
    ds = []
    for i in np.flatnonzero((kc > buoc).to_numpy()):
        a, b = t.iloc[i - 1] + buoc, t.iloc[i]                  # các mốc bị thiếu: [a, b)
        so_thieu = int((b - a) / buoc)
        cac_ngay = pd.date_range(a.normalize(), (b - pd.Timedelta(seconds=1)).normalize()).date
        if any(x.weekday() == 5 for x in cac_ngay):
            loai = 'cuoi_tuan'
        elif any(x in le for x in cac_ngay):
            loai = 'ngay_le'
        elif a.hour >= 20 and b - a <= pd.Timedelta(hours=3):
            loai = 'nghi_hang_ngay'
        else:
            loai = 'bat_thuong'
        loai_cot[i], so_thieu_cot[i] = loai, so_thieu
        ds.append({'khung': khung, 'nen_truoc': t.iloc[i - 1], 'nen_sau': b,
                   'so_nen_thieu': so_thieu, 'loai': loai})
    d['loai_khoang_trong'] = loai_cot
    d['so_nen_thieu'] = so_thieu_cot
    d['khoang_trong'] = (d['loai_khoang_trong'] == 'bat_thuong').astype(int)
    return d, ds

## Bước 4 — Nến sai cấu trúc OHLC

Một nến đúng phải có `High ≥ max(Open, Close)` và `Low ≤ min(Open, Close)`. Nến sai được
**sửa chứ không xóa** (xóa sẽ tạo khoảng trống giả):

- `High = max(Open, High, Low, Close)`, `Low = min(Open, High, Low, Close)`
- Gắn cờ `da_sua = 1` và ghi mức sai lớn nhất vào báo cáo.

In [ ]:
def sua_ohlc(d, nk):
    cao, thap = d[COT_GIA].max(axis=1), d[COT_GIA].min(axis=1)
    sai = (d['high'] < cao) | (d['low'] > thap)
    nk['sai_ohlc_da_sua'] = int(sai.sum())
    nk['sai_ohlc_lon_nhat'] = round(float(max((cao - d['high']).max(), (d['low'] - thap).max(), 0.0)), 5)
    d['high'], d['low'] = cao, thap
    d['da_sua'] = sai.astype(int)
    return d

## Chạy bước 1 → 4 cho từng khung và ghi tệp sạch

In [ ]:
LE = ngay_le(2000, 2035)
sach, nhat_ky, ds_khoang, moc_d1, tep_ra = {}, [], [], pd.Timedelta(0), []

for p in TEP_CHON:
    d = doc_tep(p)
    k = nhan_dien_khung(d)
    ten = os.path.splitext(os.path.basename(p))[0]
    print('\n══ %s — khung %s ══' % (os.path.basename(p), k))
    nk = {'khung': k, 'tep': os.path.basename(p)}
    nk['so_dong_vao'] = len(d)
    d, lech = chuan_hoa_thoi_gian(d, k, nk)          # bước 1
    if k == 'D1':
        moc_d1 = lech
    d = bo_o_trong(d, nk)                            # bước 3a (trước bước 2)
    d = bo_trung(d, nk)                              # bước 2
    d = bo_thu_bay(d, nk)                            # bước 1
    d = sua_ohlc(d, nk)                              # bước 4
    d, kh = phan_loai_khoang(d, k, LE)               # bước 3b, 3c
    d = them_ngay_giao_dich(d)                       # bước 1

    for loai in ['cuoi_tuan', 'ngay_le', 'nghi_hang_ngay', 'bat_thuong']:
        nk['khoang_' + loai] = sum(x['loai'] == loai for x in kh)
    nk['so_nen_thieu_bat_thuong'] = sum(x['so_nen_thieu'] for x in kh if x['loai'] == 'bat_thuong')
    nk['so_dong_ra'] = len(d)
    nk['tu'], nk['den'] = str(d['time'].min()), str(d['time'].max())

    assert d['time'].is_monotonic_increasing and d['time'].is_unique
    assert (d['high'] >= d[['open', 'close']].max(axis=1)).all()
    assert (d['low'] <= d[['open', 'close']].min(axis=1)).all()
    assert (d['time'].dt.dayofweek != 5).all()

    ra = d[['time'] + COT + ['ngay_giao_dich', 'loai_khoang_trong', 'so_nen_thieu', 'khoang_trong', 'da_sua']]
    p_ra = os.path.join(THU_MUC, ten + '_sach.csv')
    ra.to_csv(p_ra, index=False, date_format='%Y-%m-%d %H:%M:%S')
    tep_ra.append(p_ra)
    if k in sach:
        print('  ⚠ Đã có một tệp khung %s trước đó — kiểm tra chéo dùng tệp mới nhất.' % k)
    sach[k], ds_khoang = ra, ds_khoang + kh
    nhat_ky.append(nk)
    print('  %d dòng vào → %d dòng ra | %s → %s' % (nk['so_dong_vao'], nk['so_dong_ra'], nk['tu'], nk['den']))
    print('  Bỏ: thời gian lỗi %d, ô trống/giá lỗi %d, trùng %d + %d, thứ Bảy %d | lệch mốc %d | sửa OHLC %d'
          % (nk['thoi_gian_loi'], nk['o_trong_gia_khong_hop_le'], nk['trung_hoan_toan'],
             nk['trung_thoi_gian_khac_gia'], nk['thu_bay'], nk['lech_moc'], nk['sai_ohlc_da_sua']))
    print('  Khoảng trống: cuối tuần %d, ngày lễ %d, nghỉ hằng ngày %d, BẤT THƯỜNG %d (%d nến)'
          % (nk['khoang_cuoi_tuan'], nk['khoang_ngay_le'], nk['khoang_nghi_hang_ngay'],
             nk['khoang_bat_thuong'], nk['so_nen_thieu_bat_thuong']))
    print('  → %s' % p_ra)

if TAI_VE_MAY and _files is not None:           # tải tệp sạch về máy tính
    for p_ra in tep_ra:
        _files.download(p_ra)

## Bước 6 — Kiểm tra chéo giữa các khung

Cả 4 khung đều gộp từ cùng dữ liệu M1 của Dukascopy, nên gộp khung nhỏ lên khung lớn phải
ra **đúng** khung lớn: High của nến H4 bằng High lớn nhất của 4 nến H1 bên trong, tương
tự với Open, Low, Close, Volume. Nến lệch nghĩa là một trong hai tệp có vấn đề (hoặc vừa bị
bỏ/sửa ở bước trên) — xem bảng dưới.

In [ ]:
def gop(d, quy_tac, lech):
    g = d.set_index('time')[COT].resample(quy_tac, offset=lech)
    return g.agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'}).dropna()


kiem_cheo = []
for nho, lon, quy_tac, lech in [('M30', 'H1', '1h', pd.Timedelta(0)), ('H1', 'H4', '4h', pd.Timedelta(0)),
                                ('H1', 'D1', '24h', moc_d1)]:
    if nho not in sach or lon not in sach:
        continue
    g, l = gop(sach[nho], quy_tac, lech), sach[lon].set_index('time')[COT]
    chung = g.index.intersection(l.index)
    lech_gia = (g.loc[chung, COT_GIA] - l.loc[chung, COT_GIA]).abs()
    lech_vol = (g.loc[chung, 'volume'] - l.loc[chung, 'volume']).abs()
    kiem_cheo.append({
        'gop_tu': nho, 'so_voi': lon, 'nen_chung': len(chung),
        'nen_lech_gia': int((lech_gia > 1e-6).any(axis=1).sum()),
        'lech_gia_lon_nhat': round(float(lech_gia.max().max()), 5) if len(chung) else 0.0,
        'nen_lech_volume': int((lech_vol > 0.01 + 1e-6 * l.loc[chung, 'volume'].abs()).sum()),
        'chi_co_o_' + 'khung_nho': len(g.index.difference(l.index)),
        'chi_co_o_' + 'khung_lon': len(l.index.difference(g.index)),
    })
kiem_cheo = pd.DataFrame(kiem_cheo)
print(kiem_cheo.to_string(index=False) if len(kiem_cheo) else 'Không đủ khung để kiểm tra chéo.')

## Báo cáo làm sạch

- `bao_cao_lam_sach.csv` — mỗi khung một dòng: số dòng vào/ra và số dòng bị ảnh hưởng ở từng bước.
- `danh_sach_khoang_trong.csv` — mọi khoảng trống kèm loại; lọc `loai == 'bat_thuong'` để xem nến mất giữa phiên.
- `kiem_tra_cheo_khung.csv` — kết quả bước 6.

In [ ]:
bao_cao = pd.DataFrame(nhat_ky).set_index('tep')
bao_cao.to_csv(os.path.join(THU_MUC, 'bao_cao_lam_sach.csv'))
khoang = pd.DataFrame(ds_khoang, columns=['khung', 'nen_truoc', 'nen_sau', 'so_nen_thieu', 'loai'])
khoang.to_csv(os.path.join(THU_MUC, 'danh_sach_khoang_trong.csv'), index=False)
kiem_cheo.to_csv(os.path.join(THU_MUC, 'kiem_tra_cheo_khung.csv'), index=False)

print(bao_cao.T.to_string())
bt = khoang[khoang['loai'] == 'bat_thuong']
print('\nKhoảng trống bất thường dài nhất:')
print(bt.sort_values('so_nen_thieu', ascending=False).head(15).to_string(index=False) if len(bt) else '  (không có)')

## Biểu đồ

1. Số khoảng trống theo loại ở từng khung, và giờ (UTC) xảy ra các khoảng trống bất thường.
2. Giá đóng cửa có đánh dấu nến đã sửa OHLC (đỏ) và nến ngay sau khoảng trống bất thường (cam).

In [ ]:
loai_ds = ['cuoi_tuan', 'ngay_le', 'nghi_hang_ngay', 'bat_thuong']
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
dem = khoang.groupby(['khung', 'loai']).size().unstack(fill_value=0).reindex(columns=loai_ds, fill_value=0)
dem = dem.reindex([k for k in KHUNG_DS if k in dem.index])
dem.plot.bar(ax=ax[0], rot=0)
ax[0].set_yscale('log'); ax[0].set_title('Số khoảng trống theo loại'); ax[0].set_ylabel('số khoảng (thang log)')
for k in ['M30', 'H1']:
    x = bt[bt['khung'] == k]
    if len(x):
        (x['nen_truoc'] + BUOC[k]).dt.hour.value_counts().sort_index().plot(ax=ax[1], marker='o', label=k)
ax[1].set_title('Khoảng trống bất thường theo giờ bắt đầu (UTC)'); ax[1].set_xlabel('giờ'); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(THU_MUC, 'hinh_lam_sach_khoang_trong.png'), dpi=120); plt.show()

k = 'H1' if 'H1' in sach else next(iter(sach))
d = sach[k]
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(d['time'], d['close'], lw=0.6, color='steelblue', label='Close %s' % k)
x = d[d['da_sua'] == 1]
ax.scatter(x['time'], x['close'], color='red', s=14, zorder=3, label='đã sửa OHLC (%d)' % len(x))
x = d[d['khoang_trong'] == 1]
ax.scatter(x['time'], x['close'], color='orange', s=14, zorder=3, label='sau khoảng trống bất thường (%d)' % len(x))
ax.set_title('XAUUSD %s sau làm sạch' % k); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(THU_MUC, 'hinh_lam_sach_gia.png'), dpi=120); plt.show()

## Dùng tệp sạch ở NB1 và NB2

Điền đường dẫn tệp sạch, ví dụ

`DUONG_DAN_DU_LIEU = '/content/drive/MyDrive/Data_NghienCuu/dukascopy_XAUUSD_H1_sach.csv'`

NB1/NB2 chỉ đọc 6 cột `time, open, high, low, close, volume`; các cột đánh dấu
(`ngay_giao_dich`, `loai_khoang_trong`, `so_nen_thieu`, `khoang_trong`, `da_sua`) được giữ
trong tệp để tra cứu và trình bày trong báo cáo.

---
# PHẦN NB1 — Chỉ báo kỹ thuật → dự báo xu hướng

Đọc tệp sạch của `KHUNG_CHAY`. Đầu ra: uptrend / sideway / downtrend (1 / 0 / −1), chưa phải lệnh.

In [ ]:
# ── Ranh giới phần: xóa biến của phần trước (như mở notebook mới), giữ cấu hình chung
import matplotlib.pyplot as _plt
_plt.close('all')
_GIU = {'KHUNG_TAI', 'KHUNG_CHAY', 'BO_QUA_TAI_NEU_DA_CO', 'CAU_HINH_THU_MUC',
        'In', 'Out', 'get_ipython', 'exit', 'quit', 'display'}
for _t in [t for t in list(globals()) if not t.startswith('_') and t not in _GIU]:
    del globals()[_t]
print('Bắt đầu %s' % 'PHẦN NB1 — Chỉ báo kỹ thuật → dự báo xu hướng')

# NB1 — Chỉ số kỹ thuật → dự báo xu hướng (uptrend / sideway / downtrend)

Nhánh **kỹ thuật** của luồng nghiên cứu. Đầu vào duy nhất là tệp **OHLCV vàng**
ở khung bất kỳ (M1 … W1). Đầu ra `gold_price_technical_signal.csv` gồm các chỉ
báo và 4 dự báo xu hướng theo luật: MA, RSI, MACD và dự báo tổng hợp (≥ 2/3 luật
đồng ý). Cột `Trend` ghi dạng chữ UPTREND / SIDEWAY / DOWNTREND.

```
                     OHLCV vàng (khung bất kỳ)
                    /                          \
   NB1 chỉ báo kỹ thuật                    NB2 3 model AI
   luật → dự báo xu hướng                  XGBoost, RF, Bi-LSTM → dự báo xu hướng
                    \                          /
        NB3 chiến lược: xu hướng → lệnh BUY / SELL / FLAT
            → backtest trên CÙNG giai đoạn → Profit, Sharpe, Max DD…
```

NB1 và NB2 chỉ **dự báo xu hướng**: **1 = uptrend (trend tăng) · 0 = sideway ·
−1 = downtrend (trend giảm)** — chưa phải lệnh giao dịch. **NB3** mới áp chiến lược để
đổi xu hướng thành lệnh **BUY / SELL / FLAT** rồi backtest.

NB1 và NB2 **độc lập với nhau**: mỗi notebook tự đọc tệp OHLCV, chạy trước hay sau
đều được. NB3 chạy sau cùng, khi đã có tệp kết quả của cả hai.

Import thư viện và nơi lưu tệp

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Nơi trao đổi tệp giữa 3 notebook ─────────────────────────────
# Mỗi notebook Colab chạy trên một máy ảo riêng: tệp NB1/NB2 tạo ra KHÔNG tự có
# mặt ở NB3. Vì vậy cả 3 notebook cùng đọc/ghi vào MỘT thư mục trên Google Drive.
THU_MUC_DRIVE = '/content/drive/MyDrive/Data_NghienCuu'

try:
    from google.colab import files, drive
    TREN_COLAB = True
except ImportError:
    files = drive = None
    TREN_COLAB = False

THU_MUC = '.'
if TREN_COLAB:
    try:
        drive.mount('/content/drive')
        THU_MUC = THU_MUC_DRIVE
    except Exception as loi:
        print('Không gắn được Google Drive (%s).' % loi)
        print('→ Dùng /content: nhớ tải tệp kết quả về và tải lên ở NB3.')
        THU_MUC = '/content'
os.makedirs(THU_MUC, exist_ok=True)
print('Thư mục trao đổi dữ liệu:', os.path.abspath(THU_MUC))


def tim_tep(ten):
    """Tìm tệp đầu vào: thư mục trao đổi → thư mục hiện tại → tải lên (Colab)."""
    for p in (os.path.join(THU_MUC, ten), ten):
        if os.path.exists(p):
            return p
    if TREN_COLAB:
        print('Chưa thấy %s trong %s — hãy tải tệp này lên:' % (ten, THU_MUC))
        up = files.upload()
        if up:
            return list(up.keys())[0]
    raise FileNotFoundError('Không tìm thấy %s. Hãy chạy notebook tạo ra tệp này trước.' % ten)


def luu_tep(bang, ten):
    p = os.path.join(THU_MUC, ten)
    bang.to_csv(p, index=False, encoding='utf-8-sig')
    print('Đã lưu: %s  (%d dòng × %d cột)' % (os.path.abspath(p), bang.shape[0], bang.shape[1]))
    return p

Nhập bộ dữ liệu OHLCV

Để trống `DUONG_DAN_DU_LIEU` thì Colab hiện nút **Choose Files** để tải tệp lên.
Để khỏi tải cùng một tệp hai lần cho NB1 và NB2, có thể đặt tệp vào Drive rồi
điền đường dẫn, ví dụ `/content/drive/MyDrive/Data_NghienCuu/xau_h1.csv`.
Nhận `.csv`, `.txt` (tách bằng dấu phẩy, `;` hoặc tab), `.xlsx`, `.parquet`.

In [ ]:
DUONG_DAN_DU_LIEU = os.path.join(CAU_HINH_THU_MUC, 'dukascopy_XAUUSD_%s_sach.csv' % KHUNG_CHAY)   # tệp sạch từ phần 01

DUONG_DAN_DU_LIEU = DUONG_DAN_DU_LIEU or os.environ.get('NCKH_DU_LIEU', '')
if DUONG_DAN_DU_LIEU:
    file_name = DUONG_DAN_DU_LIEU
elif TREN_COLAB:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
else:
    raise ValueError('Đang chạy ngoài Colab: hãy điền DUONG_DAN_DU_LIEU.')

print("Tệp dữ liệu:", file_name)

Đọc dữ liệu

In [ ]:
ten_thuong = file_name.lower()
if ten_thuong.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
elif ten_thuong.endswith(('.parquet', '.pq')):
    df = pd.read_parquet(file_name)
else:
    df = pd.read_csv(file_name)
    # Tệp xuất từ MetaTrader thường tách cột bằng tab hoặc ';' → tự dò lại
    if df.shape[1] == 1:
        df = pd.read_csv(file_name, sep=None, engine='python')

print("Kích thước dữ liệu:", df.shape)
display(df.head())

 PHẦN A — XỬ LÝ DỮ LIỆU

In [ ]:
print("Các cột trong dữ liệu:")
print(df.columns.tolist())

Chuẩn hóa tên cột và cột thời gian (dùng được cho mọi khung)

Dữ liệu vàng từ các nguồn khác nhau đặt tên cột rất khác nhau. Ô dưới tự nhận
biết mà không cần sửa tay:

- **Tên cột thời gian:** `time`, `datetime`, `date`, `timestamp`, `Gmt time`,
  `<DATE>` + `<TIME>` tách rời (kiểu MetaTrader)…
- **Định dạng thời gian:** `2025-01-02 13:00`, `2025.01.02 13:00`, `02/01/2025`,
  số giây hoặc mili-giây Unix, `20250102`, có hoặc không có múi giờ.
- **Tên cột giá:** `Open/open/<OPEN>/o`, `Close/Adj Close/price`,
  `Volume/Tick Volume/tickvol`…
- **Định dạng số:** `1183.949`, `1183,949`, `1,183.949`, `1.183,949`.

Mọi thời điểm được đưa về **UTC, không kèm múi giờ**. Khung thời gian được suy
ra từ khoảng cách phổ biến nhất giữa hai nến liền nhau.

Ô này **giống hệt nhau ở NB1 và NB2**, nên hai notebook luôn đọc cùng một tệp ra
cùng một bảng dữ liệu.

In [ ]:
def _chuan_ten(c):
    return ' '.join(str(c).strip().lower().replace('<', ' ').replace('>', ' ').replace('_', ' ').split())

# Tên cột theo thứ tự ưu tiên
BI_DANH = {
    'time':   ['datetime', 'date time', 'timestamp', 'time', 'date', 'gmt time', 'local time',
               'time (utc)', 'datetime utc', 'open time', 'opentime', 'thoi gian', 'ngay'],
    'open':   ['open', 'o', 'open price', 'gia mo'],
    'high':   ['high', 'h', 'high price', 'gia cao'],
    'low':    ['low', 'l', 'low price', 'gia thap'],
    'close':  ['close', 'c', 'close price', 'adj close', 'price', 'last', 'gia dong'],
    'volume': ['volume', 'vol', 'tick volume', 'tickvol', 'real volume', 'khoi luong'],
}

def _tim_cot(cac_cot, loai):
    ten = {_chuan_ten(x): x for x in cac_cot}
    for ung_vien in BI_DANH[loai]:
        if ung_vien in ten:
            return ten[ung_vien]
    return None

def _giong_gio(s):
    """Cột chỉ chứa giờ dạng 13:00 hoặc 13:00:00 (cột <TIME> tách rời)."""
    v = s.dropna().astype(str).str.strip().head(200)
    return len(v) > 0 and v.str.fullmatch(r'\d{1,2}:\d{2}(:\d{2})?').mean() > 0.9

def _so(s):
    """Số thực. Chấp nhận 1183.949 · 1183,949 · 1,183.949 · 1.183,949."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    s = s.astype(str).str.strip().str.replace(' ', '', regex=False)
    mau = s.head(500)
    # Dấu nào đứng SAU CÙNG là dấu thập phân; dấu còn lại là phân cách hàng nghìn
    phay_la_thap_phan = (mau.str.rfind(',') > mau.str.rfind('.')).mean() > 0.5
    if phay_la_thap_phan:
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    else:
        s = s.str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors='coerce')

def _doc_thoi_gian(s):
    """Đọc cột thời gian ở mọi định dạng thường gặp, trả về UTC không múi giờ."""
    if pd.api.types.is_numeric_dtype(s):
        v = pd.to_numeric(s, errors='coerce')
        m = v.dropna().abs().median()
        if 1e7 <= m < 1e8:                                   # dạng 20250102
            return pd.to_datetime(v.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
        don_vi = 'ms' if m > 1e11 else 's'                   # Unix mili-giây hay giây
        return pd.to_datetime(v, unit=don_vi, errors='coerce', utc=True).dt.tz_localize(None)

    s = s.astype(str).str.strip()
    t = pd.to_datetime(s, errors='coerce', utc=True)
    if t.isna().mean() > 0.01:                               # định dạng lẫn lộn → đọc từng dòng
        t = pd.to_datetime(s, errors='coerce', utc=True, format='mixed')
    ung_vien = [t]
    if s.str.contains('/').mean() > 0.5:                     # 02/01/2025: ngày-trước hay tháng-trước?
        ung_vien.append(pd.to_datetime(s, errors='coerce', utc=True, format='mixed', dayfirst=True))
    # Dữ liệu giá luôn xếp theo thời gian: chọn cách đọc ít lỗi nhất và tăng dần nhiều nhất
    diem = lambda x: (x.notna().mean(), (x.diff().dt.total_seconds() > 0).mean())
    return max(ung_vien, key=diem).dt.tz_localize(None)

KHUNG_CHUAN = [(1, 'M1'), (5, 'M5'), (15, 'M15'), (30, 'M30'), (60, 'H1'),
               (240, 'H4'), (1440, 'D1'), (10080, 'W1'), (43200, 'MN')]

def nhan_dien_khung(t):
    phut = t.sort_values().diff().dt.total_seconds().div(60)
    buoc = phut[phut > 0].mode().iloc[0]
    return min(KHUNG_CHUAN, key=lambda k: abs(np.log(k[0] / buoc)))[1], buoc


# ── 1. Cột thời gian
cac_cot = list(df.columns)
ten_chuan = {_chuan_ten(x): x for x in cac_cot}
if 'date' in ten_chuan and 'time' in ten_chuan and _giong_gio(df[ten_chuan['time']]):
    cot_tg = '%s + %s' % (ten_chuan['date'], ten_chuan['time'])
    tho_tg = df[ten_chuan['date']].astype(str).str.strip() + ' ' + df[ten_chuan['time']].astype(str).str.strip()
else:
    cot_tg = _tim_cot(cac_cot, 'time')
    if cot_tg is None:
        raise ValueError('Không tìm thấy cột thời gian trong: %s' % cac_cot)
    tho_tg = df[cot_tg]

ra = pd.DataFrame({'Date': _doc_thoi_gian(tho_tg)})

# ── 2. Cột giá và khối lượng
anh_xa = {}
for loai, ten_moi in [('open', 'Open'), ('high', 'High'), ('low', 'Low'),
                      ('close', 'Close'), ('volume', 'Volume')]:
    cot = _tim_cot(cac_cot, loai)
    anh_xa[ten_moi] = cot
    ra[ten_moi] = _so(df[cot]).values if cot is not None else np.nan

if anh_xa['Close'] is None:
    raise ValueError('Không tìm thấy cột giá đóng cửa trong: %s' % cac_cot)
for ten_moi in ('Open', 'High', 'Low'):
    if anh_xa[ten_moi] is None:
        print('⚠ Thiếu cột %s → tạm dùng giá Close.' % ten_moi)
        ra[ten_moi] = ra['Close']
if anh_xa['Volume'] is None:
    ra['Volume'] = 1.0            # nhiều nguồn Forex không có khối lượng thật

df = ra
KHUNG, BUOC_PHUT = nhan_dien_khung(df['Date'].dropna())

print('Ánh xạ cột:')
print('  %-7s ← %s' % ('Date', cot_tg))
for k, v in anh_xa.items():
    print('  %-7s ← %s' % (k, v if v is not None else '(không có)'))
print('\nKhung thời gian nhận diện: %s  (bước phổ biến %.0f phút)' % (KHUNG, BUOC_PHUT))
print('Giai đoạn: %s → %s' % (df['Date'].min(), df['Date'].max()))
print('Dòng không đọc được thời gian: %d' % df['Date'].isna().sum())

Loại bỏ dữ liệu lỗi và trùng

In [ ]:
print("Trước xử lý:", df.shape)

# Bỏ dòng thiếu thời gian hoặc giá đóng cửa
df = df.dropna(subset=['Date', 'Close'])

# Giá phải lớn hơn 0
df = df[df['Close'] > 0]

# Bỏ nến vi phạm hình học: High < Low, hoặc Close nằm ngoài [Low, High]
hop_le = (df['High'] >= df['Low']) & (df['Close'] <= df['High']) & (df['Close'] >= df['Low'])
print("Nến vi phạm hình học OHLC:", int((~hop_le).sum()))
df = df[hop_le]

# Xóa dòng trùng thời gian, sắp xếp theo thời gian
df = (
    df
    .drop_duplicates(subset=['Date'], keep='last')
    .sort_values('Date')
    .reset_index(drop=True)
)

print("Sau xử lý:", df.shape)
if len(df) < 100:
    raise ValueError('Sau khi làm sạch chỉ còn %d dòng. Kiểm tra lại ánh xạ cột ở ô trên '
                     'và định dạng số của tệp (dấu thập phân, dấu phân cách hàng nghìn).' % len(df))

Kiểm tra dữ liệu sau xử lý

In [ ]:
print("Số giá trị thiếu:")
display(df.isnull().sum())

print("\nSố dòng trùng:")
print(df.duplicated().sum())

print("\n5 dòng đầu:")
display(df.head())

print("\n5 dòng cuối:")
display(df.tail())

PHẦN B — XÂY DỰNG CHỈ BÁO KỸ THUẬT



return

In [ ]:
df['Return'] = np.log(
    df['Close'] /
    df['Close'].shift(1)
)

display(
    df[['Date', 'Close', 'Return']].head(10)
)

MA10, MA30, MA50


In [ ]:
df['MA10'] = (
    df['Close']
    .rolling(window=10)
    .mean()
)

df['MA30'] = (
    df['Close']
    .rolling(window=30)
    .mean()
)

df['MA50'] = (
    df['Close']
    .rolling(window=50)
    .mean()
)

display(
    df[
        ['Date', 'Close', 'MA10', 'MA30', 'MA50']
    ].tail(10)
)

EMA12, EMA26

In [ ]:
df['EMA12'] = (
    df['Close']
    .ewm(
        span=12,
        adjust=False
    )
    .mean()
)

df['EMA26'] = (
    df['Close']
    .ewm(
        span=26,
        adjust=False
    )
    .mean()
)

display(
    df[
        ['Date', 'Close', 'EMA12', 'EMA26']
    ].tail(10)
)

RSI14

In [ ]:
delta = df['Close'].diff()

gain = delta.clip(lower=0)

loss = -delta.clip(upper=0)

avg_gain = (
    gain
    .rolling(window=14)
    .mean()
)

avg_loss = (
    loss
    .rolling(window=14)
    .mean()
)

RS = avg_gain / avg_loss

df['RSI14'] = (
    100 -
    (
        100 / (1 + RS)
    )
)

display(
    df[
        ['Date', 'Close', 'RSI14']
    ].tail(10)
)

MACD


In [ ]:
df['MACD'] = (
    df['EMA12'] -
    df['EMA26']
)

df['MACD_Signal'] = (
    df['MACD']
    .ewm(
        span=9,
        adjust=False
    )
    .mean()
)

df['MACD_Hist'] = (
    df['MACD'] -
    df['MACD_Signal']
)

display(
    df[
        [
            'Date',
            'MACD',
            'MACD_Signal',
            'MACD_Hist'
        ]
    ].tail(10)
)

Bollinger Bands

In [ ]:
df['MA20'] = (
    df['Close']
    .rolling(window=20)
    .mean()
)

df['STD20'] = (
    df['Close']
    .rolling(window=20)
    .std()
)

df['BB_Upper'] = (
    df['MA20'] +
    2 * df['STD20']
)

df['BB_Lower'] = (
    df['MA20'] -
    2 * df['STD20']
)

display(
    df[
        [
            'Date',
            'Close',
            'MA20',
            'BB_Upper',
            'BB_Lower'
        ]
    ].tail(10)
)

Volatility20

In [ ]:
df['Volatility20'] = (
    df['Return']
    .rolling(window=20)
    .std()
)

display(
    df[
        [
            'Date',
            'Return',
            'Volatility20'
        ]
    ].tail(10)
)

PHẦN C — TẠO 3 TÍN HIỆU

Tín hiệu MA

In [ ]:
df['MA_Signal'] = 0

# Dự báo TREND TĂNG (1)
df.loc[
    df['MA10'] > df['MA30'],
    'MA_Signal'
] = 1

# Dự báo TREND GIẢM (-1)
df.loc[
    df['MA10'] < df['MA30'],
    'MA_Signal'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'MA10',
            'MA30',
            'MA_Signal'
        ]
    ].tail(20)
)

Tín hiệu RSI

In [ ]:
df['RSI_Signal'] = 0

# Dự báo TREND TĂNG (1)
df.loc[
    df['RSI14'] < 30,
    'RSI_Signal'
] = 1

# Dự báo TREND GIẢM (-1)
df.loc[
    df['RSI14'] > 70,
    'RSI_Signal'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'RSI14',
            'RSI_Signal'
        ]
    ].tail(20)
)

Tín hiệu MACD

In [ ]:
df['MACD_Diff'] = (
    df['MACD'] -
    df['MACD_Signal']
)

df['MACD_Signal_Final'] = 0


# MACD cắt lên
bullish_cross = (
    (df['MACD_Diff'] > 0) &
    (df['MACD_Diff'].shift(1) <= 0)
)


# MACD cắt xuống
bearish_cross = (
    (df['MACD_Diff'] < 0) &
    (df['MACD_Diff'].shift(1) >= 0)
)


df.loc[
    bullish_cross,
    'MACD_Signal_Final'
] = 1


df.loc[
    bearish_cross,
    'MACD_Signal_Final'
] = -1

In [ ]:
display(
    df[
        [
            'Date',
            'MACD',
            'MACD_Signal',
            'MACD_Signal_Final'
        ]
    ].tail(30)
)

PHẦN D — TỔNG HỢP 3 TÍN HIỆU

Đếm số luật dự báo TREND TĂNG và TREND GIẢM

In [ ]:
df['Up_Count'] = (
    (df['MA_Signal'] == 1).astype(int)
    +
    (df['RSI_Signal'] == 1).astype(int)
    +
    (df['MACD_Signal_Final'] == 1).astype(int)
)


df['Down_Count'] = (
    (df['MA_Signal'] == -1).astype(int)
    +
    (df['RSI_Signal'] == -1).astype(int)
    +
    (df['MACD_Signal_Final'] == -1).astype(int)
)

Tạo kết quả cuối cùng -1 / 0 / 1

In [ ]:
df['Signal'] = 0


# Có ít nhất 2 luật dự báo trend tăng → UPTREND
df.loc[
    df['Up_Count'] >= 2,
    'Signal'
] = 1


# Có ít nhất 2 luật dự báo trend giảm → DOWNTREND
df.loc[
    df['Down_Count'] >= 2,
    'Signal'
] = -1

Kiểm tra kết quả

In [ ]:
display(
    df[
        [
            'Date',
            'Close',

            'MA_Signal',
            'RSI_Signal',
            'MACD_Signal_Final',

            'Up_Count',
            'Down_Count',

            'Signal'
        ]
    ].tail(50)
)

PHẦN E — ĐỔI RA UPTREND / SIDEWAY / DOWNTREND ĐỂ DỄ ĐỌC

Đây là **dự báo xu hướng**, chưa phải lệnh. NB3 mới đổi xu hướng thành lệnh BUY / SELL / FLAT theo chiến lược.


In [ ]:
df['Trend'] = df['Signal'].map({
    -1: 'DOWNTREND',
     0: 'SIDEWAY',
     1: 'UPTREND'
})

display(
    df[
        [
            'Date',
            'Close',
            'Signal',
            'Trend'
        ]
    ].tail(30)
)

PHẦN F — THỐNG KÊ KẾT QUẢ

In [ ]:
print("Số nến theo từng xu hướng dự báo:")

print(
    df['Trend'].value_counts()
)

Tỷ lệ UPTREND / SIDEWAY / DOWNTREND


In [ ]:
signal_percentage = (
    df['Trend']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Tỷ lệ xu hướng dự báo (%):")

print(signal_percentage)

PHẦN G — VẼ BIỂU ĐỒ

Giá và MA

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    df['Date'],
    df['Close'],
    label='Close'
)

plt.plot(
    df['Date'],
    df['MA10'],
    label='MA10'
)

plt.plot(
    df['Date'],
    df['MA30'],
    label='MA30'
)

plt.plot(
    df['Date'],
    df['MA50'],
    label='MA50'
)

plt.title('Gold Price and Moving Averages')

plt.xlabel('Date')
plt.ylabel('Price')

plt.legend()
plt.grid(True)

plt.show()

Bollinger Bands

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    df['Date'],
    df['Close'],
    label='Close'
)

plt.plot(
    df['Date'],
    df['BB_Upper'],
    label='Upper Band'
)

plt.plot(
    df['Date'],
    df['MA20'],
    label='MA20'
)

plt.plot(
    df['Date'],
    df['BB_Lower'],
    label='Lower Band'
)

plt.title('Bollinger Bands')

plt.xlabel('Date')
plt.ylabel('Price')

plt.legend()
plt.grid(True)

plt.show()

RSI


In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    df['Date'],
    df['RSI14'],
    label='RSI14'
)

plt.axhline(
    70,
    linestyle='--',
    label='Overbought 70'
)

plt.axhline(
    30,
    linestyle='--',
    label='Oversold 30'
)

plt.title('RSI(14)')

plt.xlabel('Date')
plt.ylabel('RSI')

plt.legend()
plt.grid(True)

plt.show()

PHẦN H — XUẤT FILE KẾT QUẢ

Đổi tên cột thời gian và giá về chữ thường (`time, open, high, low, close,
volume`) để NB3 đọc thống nhất với tệp của NB2.

In [ ]:
ket_qua = df.rename(columns={'Date': 'time', 'Open': 'open', 'High': 'high',
                             'Low': 'low', 'Close': 'close', 'Volume': 'volume'})

output_file = 'gold_price_technical_signal.csv'
duong_dan_ra = luu_tep(ket_qua, output_file)
print('Khung %s | %s → %s' % (KHUNG, ket_qua['time'].min(), ket_qua['time'].max()))

In [ ]:
# Tải tệp về máy (không bắt buộc nếu đã lưu trên Google Drive)
if TREN_COLAB and not duong_dan_ra.startswith('/content/drive'):
    files.download(duong_dan_ra)

---
# PHẦN NB2 — XGBoost walk-forward 2020–2025 → dự báo xu hướng

Độc lập với NB1: tự đọc lại tệp sạch, không dùng kết quả NB1. Mỗi năm 2020–2025 dự báo bằng mô hình chỉ học dữ liệu trước năm đó. Đầu ra: uptrend / sideway / downtrend.

In [ ]:
# ── Ranh giới phần: xóa biến của phần trước (như mở notebook mới), giữ cấu hình chung
import matplotlib.pyplot as _plt
_plt.close('all')
_GIU = {'KHUNG_TAI', 'KHUNG_CHAY', 'BO_QUA_TAI_NEU_DA_CO', 'CAU_HINH_THU_MUC',
        'In', 'Out', 'get_ipython', 'exit', 'quit', 'display'}
for _t in [t for t in list(globals()) if not t.startswith('_') and t not in _GIU]:
    del globals()[_t]
print('Bắt đầu %s' % 'PHẦN NB2 — XGBoost walk-forward 2020–2025 → dự báo xu hướng')

# NB2 — XGBoost walk-forward 2020–2025 → dự báo xu hướng (uptrend / sideway / downtrend)

Nhánh **AI** của luồng nghiên cứu, chạy **độc lập với NB1**. Tự đọc tệp OHLCV,
tự tính chỉ báo, tự gán nhãn, huấn luyện **XGBoost** theo **walk-forward**: mỗi năm
2020 → 2025 được dự báo bởi mô hình chỉ học dữ liệu **trước** năm đó. Ghép 6 năm dự
báo ngoài mẫu ra `gold_model_signals.csv`. Random Forest và Bi-LSTM của nhóm vẫn
giữ, bật thêm bằng `MO_HINH` nếu muốn so sánh.

```
                     OHLCV vàng (khung bất kỳ)
                    /                          \
   NB1 chỉ báo kỹ thuật                    NB2 3 model AI
   luật → dự báo xu hướng                  XGBoost, RF, Bi-LSTM → dự báo xu hướng
                    \                          /
        NB3 chiến lược: xu hướng → lệnh BUY / SELL / FLAT
            → backtest trên CÙNG giai đoạn → Profit, Sharpe, Max DD…
```

NB1 và NB2 chỉ **dự báo xu hướng**: **1 = uptrend (trend tăng) · 0 = sideway ·
−1 = downtrend (trend giảm)** — chưa phải lệnh giao dịch. **NB3** mới áp chiến lược để
đổi xu hướng thành lệnh **BUY / SELL / FLAT** rồi backtest.

NB1 và NB2 **độc lập với nhau**: mỗi notebook tự đọc tệp OHLCV, chạy trước hay sau
đều được. NB3 chạy sau cùng, khi đã có tệp kết quả của cả hai.

Import thư viện và nơi lưu tệp

In [ ]:
import os, json
import pandas as pd
import numpy as np

# ── Nơi trao đổi tệp giữa 3 notebook ─────────────────────────────
# Mỗi notebook Colab chạy trên một máy ảo riêng: tệp NB1/NB2 tạo ra KHÔNG tự có
# mặt ở NB3. Vì vậy cả 3 notebook cùng đọc/ghi vào MỘT thư mục trên Google Drive.
THU_MUC_DRIVE = '/content/drive/MyDrive/Data_NghienCuu'

try:
    from google.colab import files, drive
    TREN_COLAB = True
except ImportError:
    files = drive = None
    TREN_COLAB = False

THU_MUC = '.'
if TREN_COLAB:
    try:
        drive.mount('/content/drive')
        THU_MUC = THU_MUC_DRIVE
    except Exception as loi:
        print('Không gắn được Google Drive (%s).' % loi)
        print('→ Dùng /content: nhớ tải tệp kết quả về và tải lên ở NB3.')
        THU_MUC = '/content'
os.makedirs(THU_MUC, exist_ok=True)
print('Thư mục trao đổi dữ liệu:', os.path.abspath(THU_MUC))


def tim_tep(ten):
    """Tìm tệp đầu vào: thư mục trao đổi → thư mục hiện tại → tải lên (Colab)."""
    for p in (os.path.join(THU_MUC, ten), ten):
        if os.path.exists(p):
            return p
    if TREN_COLAB:
        print('Chưa thấy %s trong %s — hãy tải tệp này lên:' % (ten, THU_MUC))
        up = files.upload()
        if up:
            return list(up.keys())[0]
    raise FileNotFoundError('Không tìm thấy %s. Hãy chạy notebook tạo ra tệp này trước.' % ten)


def luu_tep(bang, ten):
    p = os.path.join(THU_MUC, ten)
    bang.to_csv(p, index=False, encoding='utf-8-sig')
    print('Đã lưu: %s  (%d dòng × %d cột)' % (os.path.abspath(p), bang.shape[0], bang.shape[1]))
    return p

Nhập bộ dữ liệu OHLCV

Để trống `DUONG_DAN_DU_LIEU` thì Colab hiện nút **Choose Files** để tải tệp lên.
Để khỏi tải cùng một tệp hai lần cho NB1 và NB2, có thể đặt tệp vào Drive rồi
điền đường dẫn, ví dụ `/content/drive/MyDrive/Data_NghienCuu/xau_h1.csv`.
Nhận `.csv`, `.txt` (tách bằng dấu phẩy, `;` hoặc tab), `.xlsx`, `.parquet`.

In [ ]:
DUONG_DAN_DU_LIEU = os.path.join(CAU_HINH_THU_MUC, 'dukascopy_XAUUSD_%s_sach.csv' % KHUNG_CHAY)   # tệp sạch từ phần 01

DUONG_DAN_DU_LIEU = DUONG_DAN_DU_LIEU or os.environ.get('NCKH_DU_LIEU', '')
if DUONG_DAN_DU_LIEU:
    file_name = DUONG_DAN_DU_LIEU
elif TREN_COLAB:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
else:
    raise ValueError('Đang chạy ngoài Colab: hãy điền DUONG_DAN_DU_LIEU.')

print("Tệp dữ liệu:", file_name)

Đọc dữ liệu

In [ ]:
ten_thuong = file_name.lower()
if ten_thuong.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
elif ten_thuong.endswith(('.parquet', '.pq')):
    df = pd.read_parquet(file_name)
else:
    df = pd.read_csv(file_name)
    # Tệp xuất từ MetaTrader thường tách cột bằng tab hoặc ';' → tự dò lại
    if df.shape[1] == 1:
        df = pd.read_csv(file_name, sep=None, engine='python')

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Chuẩn hóa tên cột và cột thời gian (dùng được cho mọi khung)

Dữ liệu vàng từ các nguồn khác nhau đặt tên cột rất khác nhau. Ô dưới tự nhận
biết mà không cần sửa tay:

- **Tên cột thời gian:** `time`, `datetime`, `date`, `timestamp`, `Gmt time`,
  `<DATE>` + `<TIME>` tách rời (kiểu MetaTrader)…
- **Định dạng thời gian:** `2025-01-02 13:00`, `2025.01.02 13:00`, `02/01/2025`,
  số giây hoặc mili-giây Unix, `20250102`, có hoặc không có múi giờ.
- **Tên cột giá:** `Open/open/<OPEN>/o`, `Close/Adj Close/price`,
  `Volume/Tick Volume/tickvol`…
- **Định dạng số:** `1183.949`, `1183,949`, `1,183.949`, `1.183,949`.

Mọi thời điểm được đưa về **UTC, không kèm múi giờ**. Khung thời gian được suy
ra từ khoảng cách phổ biến nhất giữa hai nến liền nhau.

Ô này **giống hệt nhau ở NB1 và NB2**, nên hai notebook luôn đọc cùng một tệp ra
cùng một bảng dữ liệu.

In [ ]:
def _chuan_ten(c):
    return ' '.join(str(c).strip().lower().replace('<', ' ').replace('>', ' ').replace('_', ' ').split())

# Tên cột theo thứ tự ưu tiên
BI_DANH = {
    'time':   ['datetime', 'date time', 'timestamp', 'time', 'date', 'gmt time', 'local time',
               'time (utc)', 'datetime utc', 'open time', 'opentime', 'thoi gian', 'ngay'],
    'open':   ['open', 'o', 'open price', 'gia mo'],
    'high':   ['high', 'h', 'high price', 'gia cao'],
    'low':    ['low', 'l', 'low price', 'gia thap'],
    'close':  ['close', 'c', 'close price', 'adj close', 'price', 'last', 'gia dong'],
    'volume': ['volume', 'vol', 'tick volume', 'tickvol', 'real volume', 'khoi luong'],
}

def _tim_cot(cac_cot, loai):
    ten = {_chuan_ten(x): x for x in cac_cot}
    for ung_vien in BI_DANH[loai]:
        if ung_vien in ten:
            return ten[ung_vien]
    return None

def _giong_gio(s):
    """Cột chỉ chứa giờ dạng 13:00 hoặc 13:00:00 (cột <TIME> tách rời)."""
    v = s.dropna().astype(str).str.strip().head(200)
    return len(v) > 0 and v.str.fullmatch(r'\d{1,2}:\d{2}(:\d{2})?').mean() > 0.9

def _so(s):
    """Số thực. Chấp nhận 1183.949 · 1183,949 · 1,183.949 · 1.183,949."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    s = s.astype(str).str.strip().str.replace(' ', '', regex=False)
    mau = s.head(500)
    # Dấu nào đứng SAU CÙNG là dấu thập phân; dấu còn lại là phân cách hàng nghìn
    phay_la_thap_phan = (mau.str.rfind(',') > mau.str.rfind('.')).mean() > 0.5
    if phay_la_thap_phan:
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    else:
        s = s.str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors='coerce')

def _doc_thoi_gian(s):
    """Đọc cột thời gian ở mọi định dạng thường gặp, trả về UTC không múi giờ."""
    if pd.api.types.is_numeric_dtype(s):
        v = pd.to_numeric(s, errors='coerce')
        m = v.dropna().abs().median()
        if 1e7 <= m < 1e8:                                   # dạng 20250102
            return pd.to_datetime(v.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
        don_vi = 'ms' if m > 1e11 else 's'                   # Unix mili-giây hay giây
        return pd.to_datetime(v, unit=don_vi, errors='coerce', utc=True).dt.tz_localize(None)

    s = s.astype(str).str.strip()
    t = pd.to_datetime(s, errors='coerce', utc=True)
    if t.isna().mean() > 0.01:                               # định dạng lẫn lộn → đọc từng dòng
        t = pd.to_datetime(s, errors='coerce', utc=True, format='mixed')
    ung_vien = [t]
    if s.str.contains('/').mean() > 0.5:                     # 02/01/2025: ngày-trước hay tháng-trước?
        ung_vien.append(pd.to_datetime(s, errors='coerce', utc=True, format='mixed', dayfirst=True))
    # Dữ liệu giá luôn xếp theo thời gian: chọn cách đọc ít lỗi nhất và tăng dần nhiều nhất
    diem = lambda x: (x.notna().mean(), (x.diff().dt.total_seconds() > 0).mean())
    return max(ung_vien, key=diem).dt.tz_localize(None)

KHUNG_CHUAN = [(1, 'M1'), (5, 'M5'), (15, 'M15'), (30, 'M30'), (60, 'H1'),
               (240, 'H4'), (1440, 'D1'), (10080, 'W1'), (43200, 'MN')]

def nhan_dien_khung(t):
    phut = t.sort_values().diff().dt.total_seconds().div(60)
    buoc = phut[phut > 0].mode().iloc[0]
    return min(KHUNG_CHUAN, key=lambda k: abs(np.log(k[0] / buoc)))[1], buoc


# ── 1. Cột thời gian
cac_cot = list(df.columns)
ten_chuan = {_chuan_ten(x): x for x in cac_cot}
if 'date' in ten_chuan and 'time' in ten_chuan and _giong_gio(df[ten_chuan['time']]):
    cot_tg = '%s + %s' % (ten_chuan['date'], ten_chuan['time'])
    tho_tg = df[ten_chuan['date']].astype(str).str.strip() + ' ' + df[ten_chuan['time']].astype(str).str.strip()
else:
    cot_tg = _tim_cot(cac_cot, 'time')
    if cot_tg is None:
        raise ValueError('Không tìm thấy cột thời gian trong: %s' % cac_cot)
    tho_tg = df[cot_tg]

ra = pd.DataFrame({'Date': _doc_thoi_gian(tho_tg)})

# ── 2. Cột giá và khối lượng
anh_xa = {}
for loai, ten_moi in [('open', 'Open'), ('high', 'High'), ('low', 'Low'),
                      ('close', 'Close'), ('volume', 'Volume')]:
    cot = _tim_cot(cac_cot, loai)
    anh_xa[ten_moi] = cot
    ra[ten_moi] = _so(df[cot]).values if cot is not None else np.nan

if anh_xa['Close'] is None:
    raise ValueError('Không tìm thấy cột giá đóng cửa trong: %s' % cac_cot)
for ten_moi in ('Open', 'High', 'Low'):
    if anh_xa[ten_moi] is None:
        print('⚠ Thiếu cột %s → tạm dùng giá Close.' % ten_moi)
        ra[ten_moi] = ra['Close']
if anh_xa['Volume'] is None:
    ra['Volume'] = 1.0            # nhiều nguồn Forex không có khối lượng thật

df = ra
KHUNG, BUOC_PHUT = nhan_dien_khung(df['Date'].dropna())

print('Ánh xạ cột:')
print('  %-7s ← %s' % ('Date', cot_tg))
for k, v in anh_xa.items():
    print('  %-7s ← %s' % (k, v if v is not None else '(không có)'))
print('\nKhung thời gian nhận diện: %s  (bước phổ biến %.0f phút)' % (KHUNG, BUOC_PHUT))
print('Giai đoạn: %s → %s' % (df['Date'].min(), df['Date'].max()))
print('Dòng không đọc được thời gian: %d' % df['Date'].isna().sum())

Loại bỏ dữ liệu lỗi và trùng

In [ ]:
print("Trước xử lý:", df.shape)

# Bỏ dòng thiếu thời gian hoặc giá đóng cửa
df = df.dropna(subset=['Date', 'Close'])

# Giá phải lớn hơn 0
df = df[df['Close'] > 0]

# Bỏ nến vi phạm hình học: High < Low, hoặc Close nằm ngoài [Low, High]
hop_le = (df['High'] >= df['Low']) & (df['Close'] <= df['High']) & (df['Close'] >= df['Low'])
print("Nến vi phạm hình học OHLC:", int((~hop_le).sum()))
df = df[hop_le]

# Xóa dòng trùng thời gian, sắp xếp theo thời gian
df = (
    df
    .drop_duplicates(subset=['Date'], keep='last')
    .sort_values('Date')
    .reset_index(drop=True)
)

print("Sau xử lý:", df.shape)
if len(df) < 100:
    raise ValueError('Sau khi làm sạch chỉ còn %d dòng. Kiểm tra lại ánh xạ cột ở ô trên '
                     'và định dạng số của tệp (dấu thập phân, dấu phân cách hàng nghìn).' % len(df))

Tính chỉ báo kỹ thuật (cùng công thức với NB1)

Model dùng **đúng bộ chỉ báo của NB1** (MA, EMA, RSI, MACD, Bollinger, biến động),
tính lại tại đây để NB2 chạy độc lập. Nhờ vậy phép so sánh ở NB3 là công bằng:
**cùng một lượng thông tin**, chỉ khác cách ra quyết định — luật cố định (NB1)
hay model học từ dữ liệu (NB2).

In [ ]:
C = df['Close']
df['Return'] = np.log(C / C.shift(1))
df['MA10'] = C.rolling(window=10).mean()
df['MA30'] = C.rolling(window=30).mean()
df['MA50'] = C.rolling(window=50).mean()
df['EMA12'] = C.ewm(span=12, adjust=False).mean()
df['EMA26'] = C.ewm(span=26, adjust=False).mean()

delta = C.diff()
avg_gain = delta.clip(lower=0).rolling(window=14).mean()
avg_loss = (-delta.clip(upper=0)).rolling(window=14).mean()
df['RSI14'] = 100 - 100 / (1 + avg_gain / avg_loss)

df['MACD'] = df['EMA12'] - df['EMA26']
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

df['MA20'] = C.rolling(window=20).mean()
df['STD20'] = C.rolling(window=20).std()
df['BB_Upper'] = df['MA20'] + 2 * df['STD20']
df['BB_Lower'] = df['MA20'] - 2 * df['STD20']
df['Volatility20'] = df['Return'].rolling(window=20).std()
print('Đã tính %d chỉ báo.' % (df.shape[1] - 6))

Chuẩn bị đặc trưng cho model

**Mọi chỉ báo có đơn vị USD được chia cho giá đóng cửa.** Vàng đi từ khoảng
1.050 USD (2015) lên hơn 4.500 USD (2025). Model dạng cây không ngoại suy được:
nếu học trên `MA10 = 1.800` rồi gặp `MA10 = 4.000` ở giai đoạn kiểm tra, nó chỉ
trả về vùng giá cao nhất từng thấy. Đổi sang khoảng cách tương đối (ví dụ
`MA10 / Close − 1`) giúp thước đo so sánh được qua mọi mức giá.

In [ ]:
dac_trung = pd.DataFrame(index=df.index)

# Chỉ báo có đơn vị giá → khoảng cách tương đối so với giá đóng cửa
for cot in ['MA10', 'MA30', 'MA50', 'EMA12', 'EMA26', 'MA20', 'BB_Upper', 'BB_Lower']:
    dac_trung['kc_' + cot.lower()] = df[cot] / C - 1
for cot in ['MACD', 'MACD_Signal', 'MACD_Hist', 'STD20']:
    dac_trung[cot.lower() + '_tuong_doi'] = df[cot] / C

# Chỉ báo vốn đã không phụ thuộc mức giá
dac_trung['rsi14'] = df['RSI14'] / 100
dac_trung['return'] = df['Return']
dac_trung['volatility20'] = df['Volatility20']
dac_trung['bb_vi_tri'] = (C - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

dac_trung = dac_trung.replace([np.inf, -np.inf], np.nan)
feature_cols = list(dac_trung.columns)
print('Số đặc trưng đưa vào model: %d' % len(feature_cols))
print(', '.join(feature_cols))

Walk-forward 2020 → 2025 (cửa sổ huấn luyện mở rộng dần)

Mỗi **năm kiểm tra Y**: huấn luyện trên **toàn bộ dữ liệu trước 01/01/Y**, rồi dự báo cả
năm Y. Sang năm sau, mô hình được huấn luyện lại, có thêm năm vừa qua. Ghép các năm dự báo
lại → **6 năm kiểm tra ngoài mẫu liên tiếp** cho NB3.

| Năm kiểm tra | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|---|
| Huấn luyện | 2015 → 2019 | 2015 → 2020 | 2015 → 2021 | 2015 → 2022 | 2015 → 2023 | 2015 → 2024 |

Trong **mỗi lần**: dò lại tham số nhãn `p` chỉ trên tập huấn luyện của lần đó, và bỏ `H`
nến cuối của tập huấn luyện (purging) vì nhãn của chúng đã nhìn sang năm kiểm tra.
Dữ liệu không phủ 2020–2025 → tự chuyển về một lần chia 80 % / 20 %.

In [ ]:
NAM_KIEM_TRA = [2020, 2021, 2022, 2023, 2024, 2025]
MO_HINH = ['xgb']          # thêm 'rf', 'lstm' để chạy cả Random Forest, Bi-LSTM của nhóm (chậm hơn)

nam = df['Date'].dt.year.to_numpy()
FOLD = [(str(y), nam < y, nam == y) for y in NAM_KIEM_TRA
        if (nam < y).sum() > 1000 and (nam == y).sum() > 100]
if not FOLD:
    la_train = np.arange(len(df)) < int(len(df) * 0.8)
    FOLD = [('20% cuối', la_train, ~la_train)]
    print('Dữ liệu không phủ 2020–2025 → một lần chia: 80 % đầu huấn luyện, 20 % cuối kiểm tra.')
print('Walk-forward: %d lần huấn luyện, mô hình %s' % (len(FOLD), MO_HINH))
for ten, tr, te in FOLD:
    print('  Kiểm tra %-9s | huấn luyện %6d nến (%s → %s) | kiểm tra %5d nến'
          % (ten, tr.sum(), df.loc[tr, 'Date'].min().date(), df.loc[tr, 'Date'].max().date(), te.sum()))

Gán nhãn −1 / 0 / 1 — đáp án cho model học

Nhãn dùng phương pháp **Triple Barrier** với rào cản theo **phần trăm giá**:

- Tại nến `t`, đặt rào trên `Close × (1 + p)` và rào dưới `Close × (1 − p)`.
- Quét tối đa `H` nến tiếp theo:
  chạm rào trên trước → **1 (trend tăng)**, chạm rào dưới trước → **−1 (trend giảm)**,
  hết `H` nến mà không chạm, hoặc chạm cả hai trong cùng một nến → **0 (sideway)**.
- `H` nến cuối cùng chưa đủ dữ liệu tương lai nên để trống.

**Tự thích nghi theo khung.** Mỗi khung có biên độ rất khác nhau (M15 đi vài chục
cent, D1 đi vài chục USD), nên `p` không đặt cứng mà được **dò tự động** sao cho
lớp sideway chiếm khoảng 25 %. Việc dò chỉ dùng **tập huấn luyện**.

In [ ]:
# Rào thời gian H (số nến tối đa) theo khung
RAO_THOI_GIAN = {'M1': 60, 'M5': 48, 'M15': 32, 'M30': 24, 'H1': 24,
                 'H4': 12, 'D1': 10, 'W1': 8, 'MN': 6}
SO_NEN_TOI_DA = RAO_THOI_GIAN.get(KHUNG, 24)
TY_LE_SIDEWAY_MUC_TIEU = 0.25
LUOI_PHAN_TRAM = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5,
                  2.0, 3.0, 4.0, 5.0, 7.5, 10.0]


def gan_nhan(cao, thap, dong, p, H):
    """Triple Barrier theo phần trăm giá. p tính theo %, H là số nến tối đa."""
    n = len(dong)
    tren, duoi = dong * (1 + p / 100), dong * (1 - p / 100)
    nhan = np.zeros(n)
    da_xong = np.zeros(n, dtype=bool)
    for k in range(1, H + 1):
        c_k = np.full(n, np.nan); c_k[:n - k] = cao[k:]
        t_k = np.full(n, np.nan); t_k[:n - k] = thap[k:]
        cham_tren, cham_duoi = c_k >= tren, t_k <= duoi
        moi_cham = ~da_xong & (cham_tren | cham_duoi)
        nhan[moi_cham & cham_tren & ~cham_duoi] = 1
        nhan[moi_cham & cham_duoi & ~cham_tren] = -1
        da_xong |= moi_cham               # chạm cả hai trong cùng một nến → giữ 0
    nhan[max(n - H, 0):] = np.nan         # chưa đủ H nến tương lai
    return nhan


cao, thap, dong = df['High'].to_numpy(float), df['Low'].to_numpy(float), df['Close'].to_numpy(float)


def phan_bo(p, n_do):
    """Tỷ lệ tăng / sideway / giảm (%) khi gán nhãn n_do nến đầu với tham số p."""
    nh = gan_nhan(cao[:n_do], thap[:n_do], dong[:n_do], p, SO_NEN_TOI_DA)
    nh = nh[~np.isnan(nh)]
    return 100 * (nh == 1).mean(), 100 * (nh == 0).mean(), 100 * (nh == -1).mean()


def do_p(n_do):
    """Dò p sao cho sideway ≈ mục tiêu, CHỈ trên n_do nến đầu (tập huấn luyện của lần đó).

    Tỷ lệ sideway theo p có dạng CHỮ U, không đơn điệu:
     - p rất nhỏ: nến kế tiếp chạm CẢ HAI rào cùng lúc → gán 0. Đó là nhiễu, không phải sideway.
     - p lớn: hết H nến mà không chạm rào nào → 0 đúng nghĩa sideway.
    Vì vậy chỉ dò trên NHÁNH PHẢI (p lớn hơn điểm đáy), rồi chia đôi khoảng để đạt đúng mục tiêu.
    """
    bang = pd.DataFrame([dict(zip(['p_%', 'tang_%', 'sideway_%', 'giam_%'], (p,) + phan_bo(p, n_do)))
                         for p in LUOI_PHAN_TRAM]).round(2)
    muc_tieu = 100 * TY_LE_SIDEWAY_MUC_TIEU
    i_day = bang['sideway_%'].idxmin()
    nhanh_phai = bang.loc[i_day:]
    vuot = nhanh_phai[nhanh_phai['sideway_%'] >= muc_tieu]
    if vuot.empty:
        return float(nhanh_phai['p_%'].iloc[-1]), bang
    j = vuot.index[0]
    thap_p, cao_p = float(bang.loc[max(j - 1, i_day), 'p_%']), float(bang.loc[j, 'p_%'])
    for _ in range(20):
        giua = (thap_p + cao_p) / 2
        if phan_bo(giua, n_do)[1] < muc_tieu:
            thap_p = giua
        else:
            cao_p = giua
    return round(cao_p, 4), bang


# Minh họa trên tập huấn luyện của lần walk-forward ĐẦU TIÊN
n_dau = int(FOLD[0][1].sum())
P_CHON, bang_do = do_p(n_dau)
print('Khung %s | rào thời gian %d nến | minh họa lần kiểm tra %s: dò trên %d nến huấn luyện'
      % (KHUNG, SO_NEN_TOI_DA, FOLD[0][0], n_dau))
display(bang_do)
tg, sw, gm = phan_bo(P_CHON, n_dau)
print('→ Chọn p = %.4f %%  →  Tăng %.1f %% | Sideway %.1f %% | Giảm %.1f %%  (mục tiêu sideway %.0f %%)'
      % (P_CHON, tg, sw, gm, 100 * TY_LE_SIDEWAY_MUC_TIEU))
print('Mỗi lần walk-forward sẽ dò lại p trên tập huấn luyện của chính lần đó.')

Vùng đệm (purging) — áp dụng trong từng lần walk-forward

Nhãn tại nến `t` nhìn tới `H` nến sau. Vì vậy trong mỗi lần, chỉ những nến huấn luyện
có `t + H` còn nằm **trước** năm kiểm tra mới được dùng; `H` nến sát ranh giới bị loại để
nhãn không "nhìn thấy" năm kiểm tra.

Chuyển đổi xác suất sang dự báo xu hướng ( -1 0 1 ) - Dùng chung cho cả 3 mô hình dưới



In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ánh xạ nhãn nội bộ để fit vào model (-1 -> 0, 0 -> 1, 1 -> 2)
LABEL_MAP = {-1: 0, 0: 1, 1: 2}
def map_to_internal(y):
    return np.vectorize(LABEL_MAP.get)(y)

def apply_dual_threshold(probs, tau=0.50, delta=0.15):
    """
    Chuyển xác suất 3 kịch bản thành dự báo xu hướng -1 (downtrend), 0 (sideway), 1 (uptrend)
    Cấu hình mặc định đang dùng cho chiến lược Swing (tau=0.50, delta=0.15)
    Cluoc scalping (tau=0.45, delta=0.10) ; Position (tau=0.60, delta=0.20)
    """
    signals = []
    for p in probs:
        p_down, p_flat, p_up = p[0], p[1], p[2]

        # Dự báo UPTREND khi xác suất tăng vượt ngưỡng tuyệt đối và chênh lệch vượt ngưỡng tách biệt
        if p_up >= tau and (p_up - p_down) >= delta:
            signals.append(1)
        # Dự báo DOWNTREND khi xác suất giảm vượt ngưỡng tuyệt đối và chênh lệch vượt ngưỡng tách biệt
        elif p_down >= tau and (p_down - p_up) >= delta:
            signals.append(-1)
        # Nếu lưỡng lự hoặc không đủ tự tin -> SIDEWAY
        else:
            signals.append(0)

    return np.array(signals)

Mô hình XGboost


In [ ]:
def run_xgboost(X_train, y_train, X_test, tau=0.50, delta=0.15):
    y_train_internal = map_to_internal(y_train)

    # Thiết lập siêu tham số chuẩn từ "Nghiên cứu khoa học"
    xgb_model = XGBClassifier(
        max_depth=3,
        learning_rate=0.03,
        n_estimators=150,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='multi:softprob',
        num_class=3,
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X_train, y_train_internal)

    # Lấy ma trận xác suất và đi qua bộ lọc Ngưỡng kép
    probs = xgb_model.predict_proba(X_test)
    return apply_dual_threshold(probs, tau, delta)

Random forest


In [ ]:
def run_random_forest(X_train, y_train, X_test, tau=0.50, delta=0.15):
    y_train_internal = map_to_internal(y_train)

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train_internal)

    probs = rf_model.predict_proba(X_test)
    return apply_dual_threshold(probs, tau, delta)

Bi-LSTM

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size=32, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(64, 3) # 32 * 2 (bidirectional)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Lấy trạng thái của nến cuối cùng trong cửa sổ
        # Trả về LOGIT: CrossEntropyLoss đã tự áp softmax bên trong,
        # áp thêm ở đây sẽ thành softmax hai lần và mô hình học rất kém.
        return self.fc(self.dropout(lstm_out[:, -1, :]))

def run_bilstm(X_train, y_train, X_test, seq_len=8, tau=0.50, delta=0.15):
    torch.manual_seed(42)                     # kết quả lặp lại được
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    y_train_int = map_to_internal(y_train)

    def create_seq(X, y=None):
        X_seq, y_seq = [], []
        for i in range(len(X) - seq_len):
            X_seq.append(X[i : i + seq_len])
            if y is not None: y_seq.append(y[i + seq_len])
        return np.array(X_seq), (np.array(y_seq) if y is not None else None)

    X_train_seq, y_train_seq = create_seq(X_train_scaled, y_train_int)
    X_test_seq, _ = create_seq(X_test_scaled)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BiLSTMModel(X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    loader = DataLoader(TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32),
                                      torch.tensor(y_train_seq, dtype=torch.long)),
                        batch_size=512, shuffle=False)

    model.train()
    for _ in range(8): # 8 epochs
        for b_X, b_y in loader:
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(b_X.to(device)), b_y.to(device))
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test_seq, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()

    seq_signals = apply_dual_threshold(probs, tau, delta)

    # Đệm 0 (sideway) cho các nến bị khuyết do thao tác trượt cửa sổ chuỗi
    final_signals = np.zeros(len(X_test), dtype=int)
    final_signals[seq_len:] = seq_signals
    return final_signals

Outputs — huấn luyện walk-forward và ghi tệp dự báo xu hướng cho NB3

Mỗi lần: dò `p` → gán nhãn → lấy nến huấn luyện (bỏ vùng đệm) → huấn luyện → dự báo năm
kiểm tra. Bảng đánh giá ghi **tỉ lệ dự báo trend đúng với nhãn** từng năm — độ chính xác
thuần của mô hình, trước khi đưa vào chiến lược ở NB3.

In [ ]:
# Ngưỡng kép để xác định xu hướng (mặc định cấu hình Swing).
# Scalping: tau=0.45, delta=0.10 | Position: tau=0.60, delta=0.20
tau_config = 0.50
delta_config = 0.15

HAM = {'xgb': ('XGBoost', lambda Xtr, ytr, Xte: run_xgboost(Xtr, ytr, Xte, tau=tau_config, delta=delta_config)),
       'rf': ('Random Forest', lambda Xtr, ytr, Xte: run_random_forest(Xtr, ytr, Xte, tau=tau_config, delta=delta_config)),
       'lstm': ('Bi-LSTM', lambda Xtr, ytr, Xte: run_bilstm(Xtr, ytr, Xte, seq_len=8, tau=tau_config, delta=delta_config))}

du_dac_trung = dac_trung.notna().all(axis=1).to_numpy()
X_all = dac_trung[feature_cols].to_numpy()
cac_phan, danh_gia = [], []
for ten, la_train, la_test in FOLD:
    n_do = int(la_train.sum())                                   # tập huấn luyện là phần đầu chuỗi
    p = do_p(n_do)[0]
    nhan = gan_nhan(cao, thap, dong, p, SO_NEN_TOI_DA)
    i_train = np.flatnonzero(la_train & du_dac_trung & ~np.isnan(nhan))
    i_train = i_train[i_train < n_do - SO_NEN_TOI_DA]            # vùng đệm purging
    i_test = np.flatnonzero(la_test & du_dac_trung)
    X_train, y_train, X_test = X_all[i_train], nhan[i_train].astype(int), X_all[i_test]

    phan = df.iloc[i_test][['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    phan['label'], phan['fold'], phan['p_nhan_%'] = nhan[i_test], ten, p
    for mh in MO_HINH:
        ten_mh, ham = HAM[mh]
        phan['sig_' + mh] = ham(X_train, y_train, X_test)
        tin, nh = phan['sig_' + mh], phan['label']
        co = tin.ne(0) & nh.notna()
        danh_gia.append({'Năm kiểm tra': ten, 'Mô hình': ten_mh, 'Nến huấn luyện': len(i_train),
                         'p nhãn (%)': p, 'Uptrend': int((tin == 1).sum()), 'Sideway': int((tin == 0).sum()),
                         'Downtrend': int((tin == -1).sum()),
                         'Dự báo trend đúng nhãn (%)': round(100 * (tin[co] == nh[co]).mean(), 2) if co.any() else np.nan})
    cac_phan.append(phan)
    print('  Năm %-9s: huấn luyện %6d nến, p = %.4f %% → %s' % (ten, len(i_train), p,
          ', '.join('%s %s' % (mh, phan['sig_' + mh].value_counts().sort_index().to_dict()) for mh in MO_HINH)))

df_test = pd.concat(cac_phan)
danh_gia = pd.DataFrame(danh_gia)
print('\n--- ĐÁNH GIÁ WALK-FORWARD (tín hiệu: -1 downtrend, 0 sideway, 1 uptrend) ---')
print(danh_gia.to_string(index=False))
luu_tep(danh_gia, 'danh_gia_walk_forward.csv')

# Ghi tệp cho NB3. Cột dự báo xu hướng có tiền tố "sig_" để NB3 nhận đúng.
ra = df_test.rename(columns={'Date': 'time', 'Open': 'open', 'High': 'high',
                             'Low': 'low', 'Close': 'close', 'Volume': 'volume'})
duong_dan_ra = luu_tep(ra, 'gold_model_signals.csv')

# Xem các nến mô hình dự báo có xu hướng (khác sideway)
co_tin = (df_test[['sig_' + mh for mh in MO_HINH]] != 0).any(axis=1)
display(df_test[co_tin][['Date', 'fold', 'label'] + ['sig_' + mh for mh in MO_HINH]].head(15))

In [ ]:
# Tải tệp về máy (không bắt buộc nếu đã lưu trên Google Drive)
if TREN_COLAB and not duong_dan_ra.startswith('/content/drive'):
    files.download(duong_dan_ra)

---
# PHẦN NB3 — 9 chiến lược → BUY / SELL → backtest 2020–2025

Ghép dự báo của NB1 và NB2 theo thời gian, áp 9 chiến lược (SL:TP theo R:R, đánh 2 chiều) cho 4 luật kỹ thuật và XGBoost, xem kết quả từng năm, đối chứng ngẫu nhiên, rồi đối đầu XGBoost với từng luật kỹ thuật.

In [ ]:
# ── Ranh giới phần: xóa biến của phần trước (như mở notebook mới), giữ cấu hình chung
import matplotlib.pyplot as _plt
_plt.close('all')
_GIU = {'KHUNG_TAI', 'KHUNG_CHAY', 'BO_QUA_TAI_NEU_DA_CO', 'CAU_HINH_THU_MUC',
        'In', 'Out', 'get_ipython', 'exit', 'quit', 'display'}
for _t in [t for t in list(globals()) if not t.startswith('_') and t not in _GIU]:
    del globals()[_t]
print('Bắt đầu %s' % 'PHẦN NB3 — 9 chiến lược → BUY / SELL → backtest 2020–2025')

# NB3 — Phân tích kỹ thuật và XGBoost: 9 chiến lược, walk-forward 2020–2025

**Câu hỏi nghiên cứu:** cùng một cách giao dịch, dự báo xu hướng của **XGBoost** hay của
**phân tích kỹ thuật** cho kết quả đầu tư tốt hơn — và có ổn định qua **nhiều năm** không?

- NB1 (kỹ thuật) và NB2 (XGBoost) chỉ cho **dự báo xu hướng**: 1 = uptrend · 0 = sideway · −1 = downtrend.
- Dự báo XGBoost là **walk-forward 2020 → 2025**: mỗi năm được dự báo bởi mô hình chỉ học dữ
  liệu **trước** năm đó → 6 năm kiểm tra ngoài mẫu liên tiếp.
- NB3 áp **9 chiến lược** (luật cố định: uptrend / sideway / downtrend thì làm gì + SL, TP,
  R:R) **y hệt** cho 5 nguồn → **45 kịch bản**; chênh lệch chỉ đến từ **chất lượng dự báo**.
- **Đầu ra lệnh chỉ có BUY = 1 và SELL = 0.** Được đánh 2 chiều (tối đa 1 BUY + 1 SELL; trừ S13).

| Nhánh | Nguồn dự báo |
|---|---|
| Phân tích kỹ thuật (NB1) | MA (MA10/MA30) · RSI (30/70) · MACD (cắt tín hiệu) · TH (tổng hợp ≥ 2/3 luật) |
| Mô hình AI (NB2) | **XGB** (XGBoost walk-forward) — thêm RF, LSTM nếu bật ở NB2 |

| Mã | Chiến lược | Uptrend | Sideway | Downtrend | Quản lý lệnh |
|---|---|---|---|---|---|
| S03 | Position theo trend | BUY | không vào | SELL | SL 3 % · 1:3 |
| S06 | Trend xác nhận 3 nến | BUY khi 3 nến liền uptrend | không vào | SELL khi 3 nến liền downtrend | SL 1 % · 1:2 |
| S07 | Bắt đầu xu hướng | BUY khi vừa chuyển sang uptrend | không vào | SELL khi vừa chuyển sang downtrend | SL 1 % · 1:2 |
| S08 | Vào lệnh khi giá hồi | BUY nếu Close < MA20 | không vào | SELL nếu Close > MA20 | SL 1 % · 1:2 |
| S09 | Trend + đánh vùng | BUY (1 % · 1:2) | BUY ở BB dưới · SELL ở BB trên (0,3 % · 1:1,5) | SELL (1 % · 1:2) | theo trạng thái |
| S12 | Đánh ngược dự báo (kiểm chứng) | SELL | không vào | BUY | SL 1 % · 1:2 |
| S13 | Luôn theo trend, đảo chiều | BUY (đang SELL → đóng rồi BUY) | giữ lệnh | SELL (đang BUY → đóng rồi SELL) | SL 2 %, không TP, không hedge |
| S14 | Trend + dời SL về hòa vốn | BUY | không vào | SELL | SL 1 % · TP 3 %; lãi 1R → SL về giá vào |
| S15 | Trend + SL kéo theo giá | BUY | không vào | SELL | SL 1 % kéo theo giá tốt nhất, không TP |

**Mốc chuẩn B0:** mua ở nến đầu, giữ tới cuối (lot 0,10). **Đối chứng B1:** mỗi nguồn được
**xáo trộn ngẫu nhiên** (giữ nguyên tỉ lệ uptrend/sideway/downtrend) rồi chạy lại — nguồn có
năng lực dự báo thật phải **thắng ngẫu nhiên**.


```
                     OHLCV vàng (khung bất kỳ)
                    /                          \
   NB1 chỉ báo kỹ thuật                    NB2 3 model AI
   luật → dự báo xu hướng                  XGBoost, RF, Bi-LSTM → dự báo xu hướng
                    \                          /
        NB3 chiến lược: xu hướng → lệnh BUY / SELL / FLAT
            → backtest trên CÙNG giai đoạn → Profit, Sharpe, Max DD…
```

NB1 và NB2 chỉ **dự báo xu hướng**: **1 = uptrend (trend tăng) · 0 = sideway ·
−1 = downtrend (trend giảm)** — chưa phải lệnh giao dịch. **NB3** mới áp chiến lược để
đổi xu hướng thành lệnh **BUY / SELL / FLAT** rồi backtest.

NB1 và NB2 **độc lập với nhau**: mỗi notebook tự đọc tệp OHLCV, chạy trước hay sau
đều được. NB3 chạy sau cùng, khi đã có tệp kết quả của cả hai.

In [ ]:
import os, io, json, time
import warnings
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# ── Nơi trao đổi tệp giữa 3 notebook ─────────────────────────────
# Mỗi notebook Colab chạy trên một máy ảo riêng: tệp NB1/NB2 tạo ra KHÔNG tự có
# mặt ở NB3. Vì vậy cả 3 notebook cùng đọc/ghi vào MỘT thư mục trên Google Drive.
THU_MUC_DRIVE = '/content/drive/MyDrive/Data_NghienCuu'

try:
    from google.colab import files, drive
    TREN_COLAB = True
except ImportError:
    files = drive = None
    TREN_COLAB = False

THU_MUC = '.'
if TREN_COLAB:
    try:
        drive.mount('/content/drive')
        THU_MUC = THU_MUC_DRIVE
    except Exception as loi:
        print('Không gắn được Google Drive (%s).' % loi)
        print('→ Dùng /content: nhớ tải tệp kết quả về và tải lên ở NB3.')
        THU_MUC = '/content'
os.makedirs(THU_MUC, exist_ok=True)
print('Thư mục trao đổi dữ liệu:', os.path.abspath(THU_MUC))


def tim_tep(ten):
    """Tìm tệp đầu vào: thư mục trao đổi → thư mục hiện tại → tải lên (Colab)."""
    for p in (os.path.join(THU_MUC, ten), ten):
        if os.path.exists(p):
            return p
    if TREN_COLAB:
        print('Chưa thấy %s trong %s — hãy tải tệp này lên:' % (ten, THU_MUC))
        up = files.upload()
        if up:
            return list(up.keys())[0]
    raise FileNotFoundError('Không tìm thấy %s. Hãy chạy notebook tạo ra tệp này trước.' % ten)


def luu_tep(bang, ten):
    p = os.path.join(THU_MUC, ten)
    bang.to_csv(p, index=False, encoding='utf-8-sig')
    print('Đã lưu: %s  (%d dòng × %d cột)' % (os.path.abspath(p), bang.shape[0], bang.shape[1]))
    return p

BƯỚC 0: THÔNG SỐ BACKTEST (giống Strategy Tester của MT5)

| Nhóm | Thông số | Mặc định |
|---|---|---|
| Tài khoản | số dư, đòn bẩy, stop-out | 10.000 USD · 1:100 · 50 % |
| Symbol XAUUSD | 1 lot, 1 point, spread | 100 oz · 0,01 USD · 30 points (0,30 USD) |
| Chi phí | commission, swap | 3,5 USD/lot/chiều · swap 0 (**điền theo sàn**) |
| Khối lượng | rủi ro mỗi lệnh | **1 % số dư** → lot = rủi ro ÷ (khoảng cách SL × 100 oz), làm tròn xuống 0,01 |
| Giữ 2 chiều | `ty_le_ky_quy_doi_ung` | 1,0 = tính đủ ký quỹ cho cả 2 lệnh (thận trọng) |

**Khớp lệnh:** dự báo chốt khi nến `t` đóng → lệnh khớp ở giá **mở** nến `t + 1`. Giá dữ
liệu là **Bid**, **Ask = Bid + spread**: BUY mua ở Ask, đóng ở Bid; SELL bán ở Bid, đóng ở
Ask. SL/TP tính theo **% giá vào lệnh**; một nến chạm cả SL lẫn TP → **SL khớp trước**;
giá mở nhảy qua SL/TP → khớp ở giá mở.

In [ ]:
@dataclass
class CauHinhMT5:
    # ── Tài khoản
    so_du_ban_dau: float = 10_000.0      # Initial deposit (USD)
    don_bay: int = 100                   # Leverage 1:100
    stop_out_phan_tram: float = 50.0     # margin level ≤ 50 % → đóng cưỡng bức lệnh lỗ nhất
    # ── Symbol XAUUSD
    ky_hieu: str = 'XAUUSD'
    kich_thuoc_hop_dong: float = 100.0   # 1 lot = 100 oz
    point: float = 0.01
    spread_points: int = 30              # 0,30 USD
    lot_min: float = 0.01
    lot_max: float = 10.0
    lot_step: float = 0.01
    # ── Chi phí
    hoa_hong_lot_1_chieu: float = 3.5
    swap_long_points: float = 0.0        # ĐIỀN THEO SÀN (points/lot/đêm)
    swap_short_points: float = 0.0       # ĐIỀN THEO SÀN
    gio_qua_dem_utc: int = 22
    ngay_swap_x3: int = 2                # Thứ Tư ×3
    # ── Khối lượng và giữ 2 chiều
    rui_ro_phan_tram: float = 1.0        # % số dư rủi ro mỗi lệnh
    ty_le_ky_quy_doi_ung: float = 1.0    # ký quỹ phần lệnh đối ứng khi giữ 2 chiều

    @property
    def usd_moi_point_moi_lot(self):
        return self.kich_thuoc_hop_dong * self.point


CH = CauHinhMT5()

# Kỳ hạn giao dịch: SL theo % giá vào lệnh, TP = SL × R:R
KY_HAN = {
    'scalping': dict(sl=0.3, rr=1.5),    # ngắn hạn  (S09 khi đánh vùng sideway)
    'swing':    dict(sl=1.0, rr=2.0),    # trung hạn (S06, S07, S08, S09, S12)
    'position': dict(sl=3.0, rr=3.0),    # dài hạn   (S03)
}
N_NGAU_NHIEN = 50           # số lần xáo trộn cho đối chứng B1 (0 = bỏ qua; 100 → lâu gấp đôi)
HAT_GIONG = 42

for k, v in asdict(CH).items():
    print('  %-22s %s' % (k, v))
for k, v in KY_HAN.items():
    print('  %-9s SL %.2f %% · R:R 1:%.1f · TP %.2f %% · thắng hòa vốn %.1f %%'
          % (k, v['sl'], v['rr'], v['sl'] * v['rr'], 100 / (1 + v['rr'])))

BƯỚC 1: NẠP VÀ GHÉP DỰ BÁO CỦA NB1 VÀ NB2

NB1 có dự báo trên **toàn bộ** dữ liệu; NB2 có dự báo **walk-forward** cho các năm kiểm tra
(2020 → 2025). Hai tệp được ghép theo thời gian và **chỉ giữ phần chung**, nên mọi kịch bản
chạy trên đúng cùng giai đoạn. Cột `xh_<nguồn>` là dự báo xu hướng của từng nguồn (1 / 0 / −1).

In [ ]:
def _doc(ten):
    d = pd.read_csv(tim_tep(ten), encoding='utf-8-sig')
    d.columns = [c.strip().lower() for c in d.columns]
    d['time'] = pd.to_datetime(d['time'])
    return d.sort_values('time').drop_duplicates('time')

ky_thuat = _doc('gold_price_technical_signal.csv')     # NB1
ai = _doc('gold_model_signals.csv')                     # NB2 (walk-forward)

# Nguồn dự báo: mã → (cột trong tệp, nhánh). Nguồn AI lấy theo cột có trong tệp NB2.
NGUON = {'MA': ('ma_signal', 'Kỹ thuật'), 'RSI': ('rsi_signal', 'Kỹ thuật'),
         'MACD': ('macd_signal_final', 'Kỹ thuật'), 'TH': ('signal', 'Kỹ thuật')}
for ma_ng, cot in [('XGB', 'sig_xgb'), ('RF', 'sig_rf'), ('LSTM', 'sig_lstm')]:
    if cot in ai.columns:
        NGUON[ma_ng] = (cot, 'AI')

cot_kt = [v[0] for v in NGUON.values() if v[1] == 'Kỹ thuật']
cot_ai = [v[0] for v in NGUON.values() if v[1] == 'AI']
nb1 = ky_thuat[['time', 'open', 'high', 'low', 'close', 'volume', 'rsi14', 'macd_hist',
                'ma20', 'bb_upper', 'bb_lower'] + cot_kt]
nb2 = ai[['time', 'close'] + [c for c in ['fold'] if c in ai.columns] + cot_ai].rename(columns={'close': 'close_nb2'})
df_master = nb1.merge(nb2, on='time', how='inner').sort_values('time').reset_index(drop=True)
if df_master.empty:
    raise ValueError('NB1 và NB2 không có mốc thời gian chung — kiểm tra lại hai tệp đầu vào.')

lech = (df_master['close'] - df_master['close_nb2']).abs() > 1e-6
print('⚠ %.1f %% số nến có giá đóng cửa KHÁC NHAU giữa NB1 và NB2.' % (100 * lech.mean())
      if lech.mean() > 0.01 else '✓ Giá đóng cửa của NB1 và NB2 khớp nhau: cùng một bộ dữ liệu.')
for ma_ng, (cot, _) in NGUON.items():
    df_master['xh_' + ma_ng] = df_master[cot].fillna(0).astype(int)
df_master = df_master.drop(columns=['close_nb2'] + cot_kt + cot_ai).rename(columns={'time': 'timestamp'})
NAM = df_master['timestamp'].dt.year

print('Giai đoạn backtest chung: %s → %s (%d nến, %d năm)\n'
      % (df_master['timestamp'].min(), df_master['timestamp'].max(), len(df_master), NAM.nunique()))
TEN_XU_HUONG = {1: 'UPTREND', 0: 'SIDEWAY', -1: 'DOWNTREND'}
print('Dự báo xu hướng của từng nguồn (số nến):')
print(pd.DataFrame({ng: df_master['xh_' + ng].map(TEN_XU_HUONG).value_counts() for ng in NGUON})
      .reindex(['UPTREND', 'SIDEWAY', 'DOWNTREND']).fillna(0).astype(int).to_string())
print('\nSố nến dự báo trend (≠ sideway) theo năm:')
print(pd.DataFrame({ng: (df_master['xh_' + ng] != 0).groupby(NAM).sum() for ng in NGUON}).to_string())

BƯỚC 2: 9 CHIẾN LƯỢC — DỰ BÁO XU HƯỚNG → Ý ĐỊNH VÀO LỆNH

`tao_lenh(ma, x, d)` đổi chuỗi dự báo `x` (1 / 0 / −1) thành, ở **mỗi nến**: ý định vào lệnh
(**+1 = BUY**, **−1 = SELL**, 0 = không mở lệnh mới) và SL %, TP % cho lệnh đó. Mọi điều
kiện chỉ dùng thông tin **đã có khi nến đóng** (dự báo, Close, MA20, Bollinger của chính
nến đó), lệnh khớp ở nến sau — không nhìn trước tương lai.

In [ ]:
SC, SW, PO = KY_HAN['scalping'], KY_HAN['swing'], KY_HAN['position']
CHIEN_LUOC = {
    'S03': dict(ten='Position theo trend',          nhom='Theo xu hướng',    quan_ly='co_dinh', **PO),
    'S06': dict(ten='Trend xác nhận 3 nến',         nhom='Lọc tín hiệu',     quan_ly='co_dinh', **SW),
    'S07': dict(ten='Bắt đầu xu hướng',             nhom='Lọc tín hiệu',     quan_ly='co_dinh', **SW),
    'S08': dict(ten='Vào lệnh khi giá hồi (MA20)',  nhom='Lọc tín hiệu',     quan_ly='co_dinh', **SW),
    'S09': dict(ten='Trend + đánh vùng sideway',    nhom='Có đánh sideway',  quan_ly='co_dinh', **SW),
    'S12': dict(ten='Đánh ngược dự báo',            nhom='Kiểm chứng',       quan_ly='co_dinh', **SW),
    'S13': dict(ten='Luôn theo trend, đảo chiều',   nhom='Quản lý lệnh',     quan_ly='dao_chieu', sl=2.0, rr=0.0),
    'S14': dict(ten='Trend + dời SL về hòa vốn',    nhom='Quản lý lệnh',     quan_ly='hoa_von', sl=1.0, rr=3.0),
    'S15': dict(ten='Trend + SL kéo theo giá',      nhom='Quản lý lệnh',     quan_ly='trailing', sl=1.0, rr=0.0, keo=1.0),
}


def _lui(a, k, dien):
    """Dịch mảng k nến về sau (giá trị của nến t−k đặt tại t)."""
    r = np.empty_like(a)
    r[:k] = dien
    r[k:] = a[:-k]
    return r


def tao_lenh(ma, x, d):
    """Dự báo xu hướng x (1/0/-1) → (ý định vào lệnh +1/-1/0, SL %, TP %) ở mỗi nến."""
    x = np.nan_to_num(np.asarray(x, dtype=float)).astype(int)
    n, cl = len(x), CHIEN_LUOC[ma]
    c = d['close'].to_numpy(float)
    ma20 = d['ma20'].to_numpy(float)
    bb_tren, bb_duoi = d['bb_upper'].to_numpy(float), d['bb_lower'].to_numpy(float)
    sl, rr = np.full(n, cl['sl']), np.full(n, cl['rr'])

    if ma in ('S03', 'S13', 'S14', 'S15'):
        y = x.copy()                                             # uptrend → BUY, downtrend → SELL
    elif ma == 'S06':                                            # 3 nến liền cùng dự báo
        y = np.where((x != 0) & (x == _lui(x, 1, 9)) & (x == _lui(x, 2, 9)), x, 0)
    elif ma == 'S07':                                            # vừa chuyển sang trend
        y = np.where((x != 0) & (x != _lui(x, 1, 0)), x, 0)
    elif ma == 'S08':                                            # giá hồi về phía MA20
        y = np.where((x == 1) & (c < ma20), 1, np.where((x == -1) & (c > ma20), -1, 0))
    elif ma == 'S09':                                            # trend + đánh vùng Bollinger khi sideway
        vung = np.where((x == 0) & (c <= bb_duoi), 1, np.where((x == 0) & (c >= bb_tren), -1, 0))
        y = np.where(x != 0, x, vung)
        sl = np.where(x != 0, SW['sl'], SC['sl'])
        rr = np.where(x != 0, SW['rr'], SC['rr'])
    elif ma == 'S12':
        y = -x                                                   # đánh ngược dự báo
    else:
        raise ValueError('Không có chiến lược %s' % ma)
    return y.astype(int), sl.astype(float), (sl * rr).astype(float)


print('%d chiến lược × %d nguồn = %d kịch bản' % (len(CHIEN_LUOC), len(NGUON), len(CHIEN_LUOC) * len(NGUON)))
for ma, cl in CHIEN_LUOC.items():
    print('  %s  %-30s %-17s SL %.1f %% · %s' % (ma, cl['ten'], cl['nhom'], cl['sl'],
          'R:R 1:%.1f' % cl['rr'] if cl['rr'] else 'không TP'))

BƯỚC 3: ENGINE BACKTEST KIỂU MT5 — ĐÁNH 2 CHIỀU, SL/TP THEO R:R

Mỗi nến, engine làm theo thứ tự thời gian:

1. **Swap qua đêm** cho các lệnh đang mở (22:00 UTC, Thứ Tư ×3).
2. **Mở lệnh ở giá mở** theo ý định của nến trước. Mỗi chiều tối đa 1 lệnh: đang có BUY thì
   bỏ qua ý định BUY mới, nhưng vẫn **mở được SELL** (và ngược lại). Lot theo **1 % rủi ro**;
   không đủ ký quỹ thì bỏ lệnh (*Not enough money*). S13: ý định ngược chiều **đóng lệnh
   cũ rồi đảo chiều**, không giữ 2 chiều.
3. **SL / TP trong nến**, xét riêng từng lệnh (chạm cả hai → SL trước; nhảy giá → khớp ở
   giá mở). Sau đó mới cập nhật SL cho nến sau: **S14** lãi đạt 1R → SL về giá vào;
   **S15** SL kéo theo giá tốt nhất.
4. **Equity, mức ký quỹ, Stop Out**: margin level ≤ 50 % → đóng lệnh đang lỗ nhiều nhất.

Hết dữ liệu mà còn lệnh → đóng ở giá cuối (`het_du_lieu`). Lịch sử lệnh ghi cột
**`lenh`: 1 = BUY, 0 = SELL**, lý do đóng (`tp`, `sl`, `hoa_von`, `trailing`, `dao_chieu`,
`stop_out`, `het_du_lieu`) và kết quả theo **R** (lãi/lỗ chia cho số tiền rủi ro lúc vào).

In [ ]:
def don_vi_swap(thoi_gian, ch):
    """Số 'đêm swap' phát sinh giữa nến i−1 và nến i (Thứ Tư ×3, bỏ Thứ Bảy và Chủ nhật)."""
    t = pd.to_datetime(pd.Series(thoi_gian)).reset_index(drop=True)
    ngay = ((t - pd.Timedelta(hours=ch.gio_qua_dem_utc)).dt.floor('D')
            - pd.Timestamp('1970-01-01')).dt.days.to_numpy()
    ra = np.zeros(len(t))
    for i in np.nonzero(np.diff(ngay) > 0)[0] + 1:
        for d in range(ngay[i - 1] + 1, ngay[i] + 1):
            thu = (d + 3) % 7                     # 01/01/1970 là Thứ Năm → Thứ Hai = 0
            if thu < 5:
                ra[i] += 3 if thu == ch.ngay_swap_x3 else 1
    return ra


def backtest(d, y, sl_pct, tp_pct, quan_ly='co_dinh', keo_pct=0.0, ch=None, dv_swap=None,
             lot_co_dinh=None, ghi_lenh=True):
    """y[i]: ý định vào lệnh chốt ở nến i (+1 BUY / -1 SELL / 0), khớp ở giá mở nến i+1.
    Trả về dict: equity, balance theo nến, bảng lệnh và thống kê giữ 2 chiều."""
    ch = ch or CH
    t = d['timestamp'].to_numpy()
    o, h, l, c = (d[k].to_numpy(float) for k in ('open', 'high', 'low', 'close'))
    n = len(d)
    y = np.asarray(y, dtype=int)
    dv_swap = don_vi_swap(t, ch) if dv_swap is None else dv_swap
    sp, hd, lev = ch.spread_points * ch.point, ch.kich_thuoc_hop_dong, ch.don_bay
    r_rui_ro, cm = ch.rui_ro_phan_tram / 100, ch.hoa_hong_lot_1_chieu
    swap_usd = {1: ch.swap_long_points * ch.point * hd, -1: ch.swap_short_points * ch.point * hd}
    doi_ung = ch.ty_le_ky_quy_doi_ung

    balance = ch.so_du_ban_dau
    vt = {1: None, -1: None}
    lenh = []
    eq, bal = np.empty(n), np.empty(n)
    tk = {'so_nen_2_chieu': 0, 'so_lan_2_chieu': 0, 'bo_lenh_ky_quy': 0}
    dem_lenh, dang_2_chieu = 0, False

    def ky_quy(gia):
        m = [p['lot'] * hd * gia / lev for p in vt.values() if p is not None]
        return max(m) + doi_ung * min(m) if len(m) == 2 else (m[0] if m else 0.0)

    def tha_noi(i, gia_bid):
        s = 0.0
        for p in vt.values():
            if p is not None:
                gia = gia_bid if p['huong'] > 0 else gia_bid + sp
                s += (gia - p['gia']) * p['huong'] * p['lot'] * hd + p['swap']
        return s

    def dong(p, i, gia, ly_do):
        nonlocal balance
        loi = (gia - p['gia']) * p['huong'] * p['lot'] * hd
        hh = cm * p['lot']
        balance += loi + p['swap'] - hh
        vt[p['huong']] = None
        if ghi_lenh:
            pnl = loi + p['swap'] - p['hh'] - hh
            lenh.append({'ma_lenh': p['ma'], 'type': 'BUY' if p['huong'] > 0 else 'SELL',
                         'lenh': 1 if p['huong'] > 0 else 0,
                         'entry_time': t[p['i']], 'exit_time': t[i], 'entry_price': p['gia'],
                         'sl_ban_dau': p['sl0'], 'tp': p['tp'], 'exit_price': gia, 'ly_do': ly_do,
                         'lot': p['lot'], 'profit': loi, 'swap': p['swap'],
                         'commission': -(p['hh'] + hh), 'pnl': pnl,
                         'R': pnl / p['rui_ro'] if p['rui_ro'] > 0 else np.nan,
                         'so_nen_giu': i - p['i']})

    def mo(i, huong, sl_p, tp_p):
        nonlocal balance, dem_lenh
        gia = o[i] + sp if huong > 0 else o[i]
        sl = gia * (1 - huong * sl_p / 100) if sl_p > 0 else None
        tp = gia * (1 + huong * tp_p / 100) if tp_p > 0 else None
        if lot_co_dinh is not None:
            lot = lot_co_dinh
        else:
            lot = np.floor(balance * r_rui_ro / (abs(gia - sl) * hd) / ch.lot_step + 1e-9) * ch.lot_step
            lot = float(min(max(lot, ch.lot_min), ch.lot_max))
        cu = ky_quy(o[i])
        m_moi = lot * hd * gia / lev
        m_sau = (max(cu, m_moi) + doi_ung * min(cu, m_moi)) if cu > 0 else m_moi
        if m_sau > balance + tha_noi(i, o[i]):
            tk['bo_lenh_ky_quy'] += 1                     # Not enough money
            return
        hh = cm * lot
        balance -= hh
        dem_lenh += 1
        vt[huong] = {'ma': dem_lenh, 'i': i, 'huong': huong, 'lot': lot, 'gia': gia, 'sl': sl,
                     'sl0': sl, 'tp': tp, 'hh': hh, 'swap': 0.0, 'tot': gia, 'hv': False, 'keo': False,
                     'rui_ro': abs(gia - sl) * lot * hd if sl is not None else 0.0}

    for i in range(n):
        # Nến không có lệnh mở và không có ý định vào lệnh: equity = balance, bỏ qua nhanh
        if vt[1] is None and vt[-1] is None and (i == 0 or y[i - 1] == 0):
            eq[i] = bal[i] = balance
            dang_2_chieu = False
            continue

        # 1. Swap
        if dv_swap[i] > 0:
            for p in vt.values():
                if p is not None:
                    p['swap'] += swap_usd[p['huong']] * p['lot'] * dv_swap[i]

        # 2. Mở lệnh ở giá mở theo ý định của nến trước
        if i > 0 and y[i - 1] != 0:
            huong = int(y[i - 1])
            if quan_ly == 'dao_chieu' and vt[-huong] is not None:
                p = vt[-huong]
                dong(p, i, o[i] if p['huong'] > 0 else o[i] + sp, 'dao_chieu')
            if vt[huong] is None:
                mo(i, huong, sl_pct[i - 1], tp_pct[i - 1])

        # 3. SL / TP trong nến, rồi cập nhật SL cho nến sau
        for huong in (1, -1):
            p = vt[huong]
            if p is None:
                continue
            ly_do_sl = 'trailing' if p['keo'] else ('hoa_von' if p['hv'] else 'sl')
            if huong > 0:
                if p['sl'] is not None and l[i] <= p['sl']:
                    dong(p, i, min(p['sl'], o[i]), ly_do_sl); continue
                if p['tp'] is not None and h[i] >= p['tp']:
                    dong(p, i, max(p['tp'], o[i]), 'tp'); continue
            else:
                if p['sl'] is not None and h[i] + sp >= p['sl']:
                    dong(p, i, max(p['sl'], o[i] + sp), ly_do_sl); continue
                if p['tp'] is not None and l[i] + sp <= p['tp']:
                    dong(p, i, min(p['tp'], o[i] + sp), 'tp'); continue
            if quan_ly == 'hoa_von' and not p['hv']:
                mot_r = abs(p['gia'] - p['sl0'])
                if (huong > 0 and h[i] >= p['gia'] + mot_r) or (huong < 0 and l[i] + sp <= p['gia'] - mot_r):
                    p['sl'], p['hv'] = p['gia'], True
            elif quan_ly == 'trailing':
                if huong > 0:
                    p['tot'] = max(p['tot'], h[i])
                    moi = p['tot'] * (1 - keo_pct / 100)
                    if moi > p['sl']:
                        p['sl'], p['keo'] = moi, True
                else:
                    p['tot'] = min(p['tot'], l[i] + sp)
                    moi = p['tot'] * (1 + keo_pct / 100)
                    if moi < p['sl']:
                        p['sl'], p['keo'] = moi, True

        # 4. Equity và Stop Out (đóng lệnh lỗ nhiều nhất trước)
        if vt[1] is not None or vt[-1] is not None:
            while True:
                e, m = balance + tha_noi(i, c[i]), ky_quy(c[i])
                if m <= 0 or 100 * e / m > ch.stop_out_phan_tram:
                    break
                mo_ds = [p for p in vt.values() if p is not None]
                lo = min(mo_ds, key=lambda p: ((c[i] if p['huong'] > 0 else c[i] + sp) - p['gia']) * p['huong'])
                dong(lo, i, c[i] if lo['huong'] > 0 else c[i] + sp, 'stop_out')
                if vt[1] is None and vt[-1] is None:
                    break
        hai = vt[1] is not None and vt[-1] is not None
        if hai:
            tk['so_nen_2_chieu'] += 1
            if not dang_2_chieu:
                tk['so_lan_2_chieu'] += 1
        dang_2_chieu = hai
        eq[i], bal[i] = balance + tha_noi(i, c[i]), balance

    for p in list(vt.values()):                             # hết dữ liệu
        if p is not None:
            dong(p, n - 1, c[n - 1] if p['huong'] > 0 else c[n - 1] + sp, 'het_du_lieu')
    eq[n - 1] = bal[n - 1] = balance
    return {'equity': eq, 'balance': bal, 'lenh': pd.DataFrame(lenh), 'thong_ke': tk}

BƯỚC 4: CHỈ SỐ ĐÁNH GIÁ

| Chỉ số | Ý nghĩa |
|---|---|
| Net Profit, Return, CAGR, Profit Factor, Sharpe, Sortino, Max DD, Recovery | như báo cáo MT5 (Sharpe/Sortino trên % thay đổi equity mỗi nến, quy năm) |
| Win rate · Win rate hòa vốn | tỉ lệ thắng thực tế so với mức cần để hòa vốn = 1 ÷ (1 + R:R) |
| Kỳ vọng (R) | trung bình mỗi lệnh lãi/lỗ bao nhiêu lần số tiền rủi ro — **> 0 là có lợi thế** |
| Số năm có lãi | trong các năm kiểm tra walk-forward, bao nhiêu năm kết thúc có lãi — **độ ổn định** |
| Cách đóng lệnh · Lãi BUY/SELL · Giữ 2 chiều | cách lệnh kết thúc, lãi tách theo chiều, mức độ hedge |

In [ ]:
def _sut_giam(chuoi):
    dinh = np.maximum.accumulate(chuoi)
    pct = (dinh - chuoi) / dinh
    return 100 * float(pct.max()), float((dinh - chuoi).max())


def _nen_moi_nam(ts):
    so_nam = max((ts.iloc[-1] - ts.iloc[0]).total_seconds() / (365.25 * 86400), 1e-9)
    return len(ts) / so_nam


def _sharpe(eq, nmn):
    r = np.diff(eq) / eq[:-1]
    sd = r.std(ddof=1) if len(r) > 1 else 0.0
    return float(r.mean() / sd * np.sqrt(nmn)) if sd > 0 else np.nan


def theo_nam(kq, ts, ch=None):
    """Return, Sharpe, Max DD và số lệnh của từng năm dương lịch."""
    ch = ch or CH
    eq = pd.Series(kq['equity'], index=pd.DatetimeIndex(ts))
    nmn = _nen_moi_nam(pd.Series(eq.index))
    ld = kq['lenh']
    so_lenh = pd.to_datetime(ld['entry_time']).dt.year.value_counts() if len(ld) else pd.Series(dtype=int)
    ra, dau = [], ch.so_du_ban_dau
    for nam, e in eq.groupby(eq.index.year):
        v = np.r_[dau, e.to_numpy()]
        ra.append({'Năm': int(nam), 'Return (%)': round(100 * (v[-1] / dau - 1), 2),
                   'Sharpe': round(_sharpe(v, nmn), 3), 'Max DD (%)': round(_sut_giam(v)[0], 2),
                   'Số lệnh': int(so_lenh.get(nam, 0))})
        dau = v[-1]
    return pd.DataFrame(ra)


def bao_cao(kq, ts, rr=0.0, ch=None):
    ch = ch or CH
    von0, eq, bal = ch.so_du_ban_dau, kq['equity'], kq['balance']
    nmn = _nen_moi_nam(ts)
    so_nam = len(eq) / nmn
    r = np.diff(eq) / eq[:-1]
    sd, sd_giam = (r.std(ddof=1) if len(r) > 1 else 0.0), np.sqrt((np.minimum(r, 0) ** 2).mean())
    mdd_pct, mdd_usd = _sut_giam(eq)
    ld = kq['lenh']
    pnl = ld['pnl'] if len(ld) else pd.Series(dtype=float)
    lai, lo = pnl[pnl > 0].sum(), pnl[pnl < 0].sum()
    net = bal[-1] - von0
    dem = ld['ly_do'].value_counts() if len(ld) else pd.Series(dtype=int)
    tk = kq['thong_ke']
    nbuy = int((ld['type'] == 'BUY').sum()) if len(ld) else 0
    tn = theo_nam(kq, ts, ch)
    return {
        'Net Profit': round(net, 2),
        'Return (%)': round(100 * net / von0, 2),
        'CAGR (%)': round(100 * ((max(bal[-1], 1e-9) / von0) ** (1 / so_nam) - 1), 2),
        'Profit Factor': round(lai / -lo, 3) if lo < 0 else np.nan,
        'Sharpe': round(r.mean() / sd * np.sqrt(nmn), 3) if sd > 0 else np.nan,
        'Sortino': round(r.mean() / sd_giam * np.sqrt(nmn), 3) if sd_giam > 0 else np.nan,
        'Max DD (%)': round(mdd_pct, 2),
        'Recovery Factor': round(net / mdd_usd, 3) if mdd_usd > 0 else np.nan,
        'Số năm có lãi': '%d / %d' % ((tn['Return (%)'] > 0).sum(), len(tn)),
        'Số lệnh': len(ld), 'BUY': nbuy, 'SELL': len(ld) - nbuy,
        'Win rate (%)': round(100 * (pnl > 0).mean(), 2) if len(ld) else np.nan,
        'Win rate hòa vốn (%)': round(100 / (1 + rr), 1) if rr > 0 else np.nan,
        'Kỳ vọng (R)': round(ld['R'].mean(), 3) if len(ld) and ld['R'].notna().any() else np.nan,
        **{'Đóng: ' + k: int(dem.get(k, 0)) for k in
           ['tp', 'sl', 'hoa_von', 'trailing', 'dao_chieu', 'stop_out', 'het_du_lieu']},
        'Lãi BUY': round(ld.loc[ld['type'] == 'BUY', 'pnl'].sum(), 2) if len(ld) else 0.0,
        'Lãi SELL': round(ld.loc[ld['type'] == 'SELL', 'pnl'].sum(), 2) if len(ld) else 0.0,
        'Giữ TB (nến)': round(ld['so_nen_giu'].mean(), 1) if len(ld) else np.nan,
        'Số lần 2 chiều': tk['so_lan_2_chieu'],
        '% nến 2 chiều': round(100 * tk['so_nen_2_chieu'] / len(eq), 2),
        'Bỏ lệnh (ký quỹ)': tk['bo_lenh_ky_quy'],
        'Total Commission': round(ld['commission'].sum(), 2) if len(ld) else 0.0,
        'Total Swap': round(ld['swap'].sum(), 2) if len(ld) else 0.0,
    }


def in_bao_cao(bc, ten):
    print('=' * 96)
    print('STRATEGY TESTER REPORT — %s | %s | %s → %s' % (ten, CH.ky_hieu, df_master['timestamp'].min(),
                                                          df_master['timestamp'].max()))
    print('=' * 96)
    k = list(bc.items())
    nua = (len(k) + 1) // 2
    for (a, x), (b, yv) in zip(k[:nua], k[nua:] + [('', '')]):
        print('  %-24s %-14s  %-24s %s' % (a, x, b, yv))

Hàm vẽ khớp lệnh của nhóm

In [ ]:
# ==============================================================================
# BƯỚC 4: VẼ CHART (CÓ TÙY BIẾN CHỈ BÁO THEO Ý NHÓM TRƯỞNG)
# ==============================================================================
def plot_results(df_result, trades_df, signal_col, indicator_col=""):
    peak = df_result['equity'].cummax()
    drawdown = (df_result['equity'] - peak) / peak * 100

    has_indicator = indicator_col in df_result.columns
    rows = 4 if has_indicator else 3

    titles = [f'Khớp Lệnh: {signal_col.upper()}']
    if has_indicator: titles.append(f'Chỉ báo: {indicator_col.upper()}')
    titles.extend(['Đường Tăng Trưởng (Equity)', 'Sụt Giảm Tài Khoản (Drawdown %)'])

    fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.05, subplot_titles=titles,
                        row_heights=[0.4, 0.2, 0.2, 0.2] if has_indicator else [0.5, 0.3, 0.2])

    fig.add_trace(go.Candlestick(x=df_result['timestamp'], open=df_result['open'], high=df_result['high'], low=df_result['low'], close=df_result['close'], name='OHLCV'), row=1, col=1)
    if not trades_df.empty:
        buys, sells = trades_df[trades_df['type'] == 'BUY'], trades_df[trades_df['type'] == 'SELL']
        if not buys.empty: fig.add_trace(go.Scatter(x=buys['entry_time'], y=buys['entry_price'], mode='markers', marker=dict(symbol='triangle-up', size=12, color='green'), name='BUY'), row=1, col=1)
        if not sells.empty: fig.add_trace(go.Scatter(x=sells['entry_time'], y=sells['entry_price'], mode='markers', marker=dict(symbol='triangle-down', size=12, color='red'), name='SELL'), row=1, col=1)

    curr_row = 2
    if has_indicator:
        fig.add_trace(go.Scatter(x=df_result['timestamp'], y=df_result[indicator_col], mode='lines', line=dict(color='purple'), name=indicator_col.upper()), row=curr_row, col=1)
        curr_row += 1

    fig.add_trace(go.Scatter(x=df_result['timestamp'], y=df_result['equity'], mode='lines', line=dict(color='blue'), name='Equity'), row=curr_row, col=1)
    fig.add_trace(go.Scatter(x=df_result['timestamp'], y=drawdown, mode='lines', fill='tozeroy', line=dict(color='red'), name='Drawdown'), row=curr_row+1, col=1)

    fig.update_layout(height=900 if has_indicator else 800, xaxis_rangeslider_visible=False, template='plotly_white')
    fig.show()

BƯỚC 5: CHẠY 45 KỊCH BẢN VÀ MỐC CHUẨN B0

Mã kịch bản dạng `S03-XGB` (chiến lược S03 trên dự báo XGBoost). Ghi ra:
`backtest_comparison_report.csv` (toàn giai đoạn), `ket_qua_theo_nam.csv` (từng năm),
`lich_su_lenh_tat_ca.csv` (mọi lệnh, cột `lenh` 1 = BUY / 0 = SELL) và
`tin_hieu_lenh_NB3.csv` (ý định vào lệnh ở từng nến: 1 = BUY, 0 = SELL, trống = không vào).

In [ ]:
DV_SWAP = don_vi_swap(df_master['timestamp'], CH)
TS = df_master['timestamp']
ket_qua, lich_su, theo_nam_ds, luu = [], [], [], {}
tin_hieu = pd.DataFrame({'timestamp': TS})
t0 = time.time()

def _ghi(kb, ma, ten, nhom, ng, nhanh, kq, rr):
    ket_qua.append({'Kịch bản': kb, 'Chiến lược': ma, 'Tên chiến lược': ten, 'Nhóm': nhom,
                    'Nguồn': ng, 'Nhánh': nhanh, **bao_cao(kq, TS, rr)})
    theo_nam_ds.append(theo_nam(kq, TS).assign(**{'Kịch bản': kb, 'Chiến lược': ma, 'Nguồn': ng, 'Nhánh': nhanh}))
    luu[kb] = kq

for ma, cl in CHIEN_LUOC.items():
    for ng in NGUON:
        y, slp, tpp = tao_lenh(ma, df_master['xh_' + ng].to_numpy(), df_master)
        kq = backtest(df_master, y, slp, tpp, cl['quan_ly'], cl.get('keo', 0.0), CH, DV_SWAP)
        kb = '%s-%s' % (ma, ng)
        _ghi(kb, ma, cl['ten'], cl['nhom'], ng, NGUON[ng][1], kq, cl['rr'])
        if len(kq['lenh']):
            lich_su.append(kq['lenh'].assign(kich_ban=kb))
        tin_hieu[kb] = pd.Series(y).map({1: 1, -1: 0}).values        # 1 = BUY, 0 = SELL, trống = không vào
    print('  %s xong (%.0f giây)' % (ma, time.time() - t0))

# Mốc chuẩn B0: mua ở nến đầu, giữ tới cuối, lot cố định 0,10, không SL/TP
y0 = np.zeros(len(df_master), dtype=int); y0[0] = 1
kq0 = backtest(df_master, y0, np.zeros(len(y0)), np.zeros(len(y0)), ch=CH, dv_swap=DV_SWAP, lot_co_dinh=0.10)
_ghi('B0', 'B0', 'Mua và giữ', 'Mốc chuẩn', '—', 'Mốc chuẩn', kq0, 0.0)

df_kq = pd.DataFrame(ket_qua)
df_nam = pd.concat(theo_nam_ds, ignore_index=True)
lich_su_lenh = pd.concat(lich_su, ignore_index=True) if lich_su else pd.DataFrame()
print('\n%d kịch bản xong sau %.0f giây.' % (len(df_kq), time.time() - t0))
cot_xem = ['Kịch bản', 'Nhánh', 'Net Profit', 'CAGR (%)', 'Profit Factor', 'Sharpe', 'Max DD (%)',
           'Số năm có lãi', 'Số lệnh', 'Win rate (%)', 'Win rate hòa vốn (%)', 'Kỳ vọng (R)']
print('\nToàn bộ kịch bản, xếp theo Sharpe:')
print(df_kq.sort_values('Sharpe', ascending=False)[cot_xem].to_string(index=False))

BƯỚC 6: ĐỐI CHỨNG B1 — DỰ BÁO NGẪU NHIÊN

Với mỗi nguồn và mỗi chiến lược, chuỗi dự báo được **xáo trộn ngẫu nhiên** `N_NGAU_NHIEN`
lần (giữ nguyên số nến uptrend / sideway / downtrend) rồi backtest lại. **Phân vị** của
kịch bản thật = bao nhiêu % lần ngẫu nhiên có Sharpe thấp hơn. Phân vị ≥ 95 % → dự báo
tốt hơn ngẫu nhiên một cách đáng tin. Mất khoảng 10 phút với `N_NGAU_NHIEN = 50`; đặt 0 ở
Bước 0 để bỏ qua.

In [ ]:
df_kq['Phân vị Sharpe (%)'] = np.nan
df_kq['Phân vị Net (%)'] = np.nan
ngau_nhien = pd.DataFrame()
if N_NGAU_NHIEN > 0:
    rng = np.random.default_rng(HAT_GIONG)
    nmn = _nen_moi_nam(TS)
    t0 = time.time()
    hang = []
    for ma, cl in CHIEN_LUOC.items():
        for ng in NGUON:
            x = df_master['xh_' + ng].to_numpy()
            sh, net = [], []
            for _ in range(N_NGAU_NHIEN):
                y, slp, tpp = tao_lenh(ma, rng.permutation(x), df_master)
                kq = backtest(df_master, y, slp, tpp, cl['quan_ly'], cl.get('keo', 0.0), CH, DV_SWAP,
                              ghi_lenh=False)
                sh.append(_sharpe(kq['equity'], nmn)); net.append(kq['balance'][-1] - CH.so_du_ban_dau)
            sh, net = np.nan_to_num(np.array(sh), nan=-np.inf), np.array(net)
            dong_kq = df_kq['Kịch bản'] == '%s-%s' % (ma, ng)
            that_sh = df_kq.loc[dong_kq, 'Sharpe'].iloc[0]
            that_net = df_kq.loc[dong_kq, 'Net Profit'].iloc[0]
            df_kq.loc[dong_kq, 'Phân vị Sharpe (%)'] = round(100 * np.mean(sh < (that_sh if pd.notna(that_sh) else -np.inf)), 1)
            df_kq.loc[dong_kq, 'Phân vị Net (%)'] = round(100 * np.mean(net < that_net), 1)
            hang.append({'Chiến lược': ma, 'Nguồn': ng,
                         'Sharpe ngẫu nhiên TB': np.nanmean(np.where(np.isfinite(sh), sh, np.nan)),
                         'Net ngẫu nhiên TB': net.mean(), 'Net ngẫu nhiên P95': np.percentile(net, 95)})
        print('  %s xong (%.0f giây)' % (ma, time.time() - t0))
    ngau_nhien = pd.DataFrame(hang)
    luu_tep(ngau_nhien, 'ket_qua_ngau_nhien.csv')
    print('\nSố chiến lược THẮNG NGẪU NHIÊN (phân vị Sharpe ≥ 95 %) theo nguồn:')
    thang = (df_kq[df_kq['Chiến lược'] != 'B0'].groupby('Nguồn')['Phân vị Sharpe (%)']
             .apply(lambda s: '%d / %d' % ((s >= 95).sum(), s.notna().sum())))
    print(thang.reindex(list(NGUON)).to_string())
else:
    print('Bỏ qua đối chứng ngẫu nhiên (N_NGAU_NHIEN = 0).')

duong_dan_bao_cao = luu_tep(df_kq, 'backtest_comparison_report.csv')
luu_tep(df_nam, 'ket_qua_theo_nam.csv')
luu_tep(lich_su_lenh, 'lich_su_lenh_tat_ca.csv')
luu_tep(tin_hieu, 'tin_hieu_lenh_NB3.csv')

BƯỚC 7: SO SÁNH PHÂN TÍCH KỸ THUẬT VÀ XGBOOST

1. **Bảng chéo chiến lược × nguồn** (Sharpe, CAGR) trên toàn giai đoạn.
2. **Từng năm:** Return của mỗi kịch bản theo năm — nguồn nào ổn định qua các năm.
3. **Đối đầu trực tiếp XGBoost với từng luật kỹ thuật** trên **9 chiến lược × các năm**
   (mỗi cặp = cùng chiến lược, cùng năm): XGB thắng bao nhiêu cặp, và **kiểm định Wilcoxon**
   ghép cặp (p < 0,05 → khác biệt có ý nghĩa thống kê, không phải may rủi).
4. **Nhánh kỹ thuật (tốt nhất / trung bình 4 luật) và nhánh AI** theo từng chiến lược.

In [ ]:
ds_ng = list(NGUON)
KT = [ng for ng in ds_ng if NGUON[ng][1] == 'Kỹ thuật']
AI_ = [ng for ng in ds_ng if NGUON[ng][1] == 'AI']
ten_cl = {ma: '%s %s' % (ma, cl['ten']) for ma, cl in CHIEN_LUOC.items()}
kq_cl = df_kq[df_kq['Chiến lược'] != 'B0']
bang_sharpe = kq_cl.pivot(index='Chiến lược', columns='Nguồn', values='Sharpe')[ds_ng].rename(index=ten_cl)
bang_cagr = kq_cl.pivot(index='Chiến lược', columns='Nguồn', values='CAGR (%)')[ds_ng].rename(index=ten_cl)


def hien_bang(b, tieu_de, dinh_dang='{:.2f}'):
    print('\n' + tieu_de)
    try:
        display(b.style.background_gradient(cmap='RdYlGn', axis=None).format(dinh_dang))
    except Exception:
        print(b.round(2).to_string())


hien_bang(bang_sharpe, 'SHARPE toàn giai đoạn — chiến lược × nguồn (kỹ thuật: %s | AI: %s)' % (' '.join(KT), ' '.join(AI_)))
hien_bang(bang_cagr, 'CAGR (%/năm) toàn giai đoạn — chiến lược × nguồn')
luu_tep(bang_sharpe.reset_index(), 'bang_sharpe_chien_luoc_x_nguon.csv')

# ── 2. Từng năm
nam_cl = df_nam[df_nam['Chiến lược'] != 'B0']
bang_nam = nam_cl.pivot_table(index=['Chiến lược', 'Nguồn'], columns='Năm', values='Return (%)')
bang_nam = bang_nam.reindex([(ma, ng) for ma in CHIEN_LUOC for ng in ds_ng])
bang_nam['Số năm lãi'] = (bang_nam > 0).sum(axis=1)
hien_bang(bang_nam, 'RETURN (%) TỪNG NĂM — mỗi dòng một kịch bản', '{:.1f}')
b0_nam = df_nam[df_nam['Chiến lược'] == 'B0'].set_index('Năm')['Return (%)']
print('\nMua và giữ (B0) theo năm: ' + ' | '.join('%d: %+.1f %%' % kv for kv in b0_nam.items()))

# ── 3. Đối đầu XGBoost (và AI khác nếu có) với từng luật kỹ thuật, trên cặp (chiến lược, năm)
try:
    from scipy.stats import wilcoxon
except ImportError:
    wilcoxon = None

def _p(dd):
    dd = dd[dd != 0]
    if wilcoxon is None or len(dd) < 6:
        return np.nan
    return wilcoxon(dd).pvalue

doi_dau = []
ret = nam_cl.pivot_table(index=['Chiến lược', 'Năm'], columns='Nguồn', values='Return (%)')
shp = nam_cl.pivot_table(index=['Chiến lược', 'Năm'], columns='Nguồn', values='Sharpe').fillna(0)  # năm không giao dịch → 0
for a in AI_:
    for k in KT:
        d_ret, d_shp = (ret[a] - ret[k]).dropna(), (shp[a] - shp[k]).dropna()
        d_toan = (bang_sharpe[a] - bang_sharpe[k]).dropna()
        doi_dau.append({'Cặp': '%s vs %s' % (a, k),
                        'Số chiến lược so được': len(d_toan),       # bỏ chiến lược mà một bên không có lệnh
                        'Thắng Sharpe toàn kỳ': int((d_toan > 0).sum()),
                        'Số cặp (chiến lược, năm)': len(d_ret),
                        'Thắng Return từng năm': int((d_ret > 0).sum()),
                        'Chênh Return TB (điểm %)': round(d_ret.mean(), 2),
                        'p Wilcoxon (Return năm)': round(_p(d_ret), 4),
                        'Thắng Sharpe từng năm': int((d_shp > 0).sum()),
                        'p Wilcoxon (Sharpe năm)': round(_p(d_shp), 4)})
doi_dau = pd.DataFrame(doi_dau)
print('\nĐỐI ĐẦU: %s với từng luật kỹ thuật (cặp = cùng chiến lược, cùng năm)' % ', '.join(AI_))
print(doi_dau.to_string(index=False))
print('p < 0,05 → khác biệt có ý nghĩa thống kê; p ≥ 0,05 → chưa đủ bằng chứng.')
luu_tep(doi_dau, 'doi_dau_xgb_vs_ky_thuat.csv')

# ── 4. Nhánh kỹ thuật và nhánh AI theo từng chiến lược
hang = []
for ma in CHIEN_LUOC:
    s = bang_sharpe.loc[ten_cl[ma]]
    hang.append({'Chiến lược': ten_cl[ma],
                 'KT tốt nhất': '%s %.2f' % (s[KT].idxmax(), s[KT].max()) if s[KT].notna().any() else '—',
                 'AI tốt nhất': '%s %.2f' % (s[AI_].idxmax(), s[AI_].max()) if s[AI_].notna().any() else '—',
                 'Sharpe KT max': s[KT].max(), 'Sharpe KT TB': s[KT].mean(),
                 'Sharpe AI max': s[AI_].max(), 'Sharpe AI TB': s[AI_].mean()})
so_sanh = pd.DataFrame(hang)
so_sanh['Thắng (so KT tốt nhất)'] = np.where(so_sanh['Sharpe AI max'] > so_sanh['Sharpe KT max'], 'AI', 'Kỹ thuật')
so_sanh['Thắng (so KT trung bình)'] = np.where(so_sanh['Sharpe AI max'] > so_sanh['Sharpe KT TB'], 'AI', 'Kỹ thuật')
print('\nTỪNG CHIẾN LƯỢC: nhánh Kỹ thuật và nhánh AI (Sharpe toàn giai đoạn)')
print(so_sanh.round(3).to_string(index=False))
print('\nAI thắng %d / %d chiến lược khi so với luật kỹ thuật TỐT NHẤT, %d / %d khi so với TRUNG BÌNH 4 luật.'
      % ((so_sanh['Thắng (so KT tốt nhất)'] == 'AI').sum(), len(so_sanh),
         (so_sanh['Thắng (so KT trung bình)'] == 'AI').sum(), len(so_sanh)))
luu_tep(so_sanh, 'so_sanh_ky_thuat_vs_ai.csv')

b0 = df_kq[df_kq['Kịch bản'] == 'B0'].iloc[0]
print('\nMốc chuẩn mua và giữ: Net %.2f USD | CAGR %.2f %% | Sharpe %.3f | Max DD %.2f %%'
      % (b0['Net Profit'], b0['CAGR (%)'], b0['Sharpe'], b0['Max DD (%)']))

BƯỚC 8: BIỂU ĐỒ VÀ BÁO CÁO CHI TIẾT MỘT KỊCH BẢN

- Bản đồ nhiệt Sharpe; Return từng năm của `CHIEN_LUOC_VE` theo nguồn.
- Equity của `CHIEN_LUOC_VE` trên cả 5 nguồn và mốc mua-và-giữ.
- Báo cáo kiểu MT5, lịch sử lệnh và biểu đồ khớp lệnh của `KICH_BAN_VE`
  (để trống: kịch bản XGBoost có Sharpe cao nhất).

In [ ]:
CHIEN_LUOC_VE = ''      # ví dụ 'S13'; để trống → chiến lược của kịch bản được chọn
KICH_BAN_VE = ''        # ví dụ 'S13-XGB'; để trống → kịch bản AI có Sharpe cao nhất
CHI_BAO_VE = 'rsi14'

fig = go.Figure(go.Heatmap(z=bang_sharpe.values, x=bang_sharpe.columns, y=bang_sharpe.index,
                           colorscale='RdYlGn', zmid=0, text=np.round(bang_sharpe.values, 2),
                           texttemplate='%{text}'))
fig.update_layout(title='Sharpe toàn giai đoạn: chiến lược × nguồn', height=520, template='plotly_white',
                  yaxis_autorange='reversed')
fig.show()

ung_vien = df_kq[df_kq['Nhánh'] == 'AI'] if (df_kq['Nhánh'] == 'AI').any() else df_kq[df_kq['Kịch bản'] != 'B0']
KICH_BAN_VE = KICH_BAN_VE or ung_vien.sort_values('Sharpe', ascending=False).iloc[0]['Kịch bản']
CHIEN_LUOC_VE = CHIEN_LUOC_VE or KICH_BAN_VE.split('-')[0]

d = df_nam[df_nam['Chiến lược'].isin([CHIEN_LUOC_VE, 'B0'])]
fig = go.Figure()
for ng in ds_ng + ['—']:
    x = d[d['Nguồn'] == ng]
    fig.add_bar(x=x['Năm'], y=x['Return (%)'], name='B0 mua và giữ' if ng == '—' else ng)
fig.update_layout(title='Return từng năm — %s' % ten_cl[CHIEN_LUOC_VE], barmode='group', height=480,
                  template='plotly_white', yaxis_title='%')
fig.show()

fig = go.Figure()
for ng in ds_ng:
    kb = '%s-%s' % (CHIEN_LUOC_VE, ng)
    fig.add_scatter(x=TS, y=luu[kb]['equity'], mode='lines', name=kb,
                    line=dict(dash='dot' if NGUON[ng][1] == 'Kỹ thuật' else 'solid',
                              width=1 if NGUON[ng][1] == 'Kỹ thuật' else 2.5))
fig.add_scatter(x=TS, y=luu['B0']['equity'], mode='lines', name='B0 mua và giữ', line=dict(dash='dash', color='black'))
fig.update_layout(title='Equity — %s (kỹ thuật: chấm · AI: liền)' % ten_cl[CHIEN_LUOC_VE],
                  height=550, template='plotly_white', yaxis_title='USD')
fig.show()

kq = luu[KICH_BAN_VE]
in_bao_cao(df_kq[df_kq['Kịch bản'] == KICH_BAN_VE].iloc[0].drop(['Kịch bản']).to_dict(), KICH_BAN_VE)
print('\nTheo năm:')
print(df_nam[df_nam['Kịch bản'] == KICH_BAN_VE].drop(columns=['Kịch bản', 'Chiến lược', 'Nguồn', 'Nhánh']).to_string(index=False))
ld = kq['lenh']
print('\nLịch sử lệnh (20 dòng đầu) — cột lenh: 1 = BUY, 0 = SELL:')
print(ld.head(20).round(3).to_string(index=False) if len(ld) else '(không có lệnh)')

df_chart = df_master[['timestamp', 'open', 'high', 'low', 'close', 'rsi14', 'macd_hist']].copy()
df_chart['equity'] = kq['equity']
plot_results(df_chart, ld if len(ld) else pd.DataFrame(columns=['type']), KICH_BAN_VE, CHI_BAO_VE)

In [ ]:
# Tải bảng so sánh về máy (không bắt buộc nếu đã lưu trên Google Drive)
if TREN_COLAB and not duong_dan_bao_cao.startswith('/content/drive'):
    files.download(duong_dan_bao_cao)